In [ ]:
!pip install -q "transformers==4.57.6" peft sentencepiece
import os
os.environ["HF_TOKEN"] = "PASTE_YOUR_TOKEN_HERE"


In [ ]:
import pathlib
for sub in ("scripts", "src/activeqa", "data/indic_rag_te", "models"):
    pathlib.Path("/kaggle/working/activeqa/" + sub).mkdir(parents=True, exist_ok=True)
print("tree ready")


In [ ]:
%%writefile /kaggle/working/activeqa/src/activeqa/__init__.py
"""CPU oriented Telugu ActiveQA research implementation."""

In [ ]:
%%writefile /kaggle/working/activeqa/src/activeqa/preprocessing.py
import re, unicodedata

def normalize(text):
    text = unicodedata.normalize("NFC", str(text or ""))
    return re.sub(r"\s+", " ", text).strip()

def tokens(text):
    out=[]; current=[]
    for ch in normalize(text).casefold():
        if unicodedata.category(ch)[0] in {"L", "N", "M"}: current.append(ch)
        elif current:
            out.append("".join(current)); current=[]
    if current: out.append("".join(current))
    return out

def entity_signature(text):
    return set(re.findall(r"\d+(?:[./:-]\d+)*|\b[A-Z][\w-]*\b", text))


_INTERROGATIVES = {
    "ఏ": "which", "ఏది": "which",
    "ఏమి": "what", "ఏమిటి": "what",
    "ఎవరు": "who", "ఎవరిని": "who", "ఎవరికి": "who", "ఎవరి": "who",
    "ఎవరిది": "who",
    "ఎప్పుడు": "when", "ఎప్పటి": "when", "ఎప్పటికి": "when",
    "ఎప్పటిది": "when",
    "ఎక్కడ": "where", "ఎక్కడి": "where", "ఎక్కడికి": "where",
    "ఎక్కడిది": "where",
    "ఎందుకు": "why", "ఎందుకని": "why",
    "ఎలా": "how", "ఎలాంటి": "how", "ఎలాంటిది": "how",
    "ఎంత": "how_much", "ఎంతకు": "how_much", "ఎంతమంది": "how_many",
    "ఎన్ని": "how_many",
}


def interrogatives(text):
    """Return the answer-type categories of the Telugu interrogatives in text."""
    return {_INTERROGATIVES[token] for token in tokens(text) if token in _INTERROGATIVES}

In [ ]:
%%writefile /kaggle/working/activeqa/src/activeqa/data.py
from __future__ import annotations
import hashlib, json
from pathlib import Path
from .preprocessing import normalize

def _row(r):
    a=r.get("answers") or {}; texts=a.get("text", []) if isinstance(a,dict) else []
    return {"id":str(r.get("id", "")), "question":normalize(r.get("question")), "context":normalize(r.get("context")), "answers":[normalize(x) for x in texts if normalize(x)]}

def _load_indicqa_json(path):
    """Flatten IndicQA's published SQuAD-style JSON structure."""
    with open(path, encoding="utf-8") as f:
        raw=json.load(f)
    rows=[]
    for article in raw.get("data", []):
        for paragraph in article.get("paragraphs", []):
            context=normalize(paragraph.get("context"))
            for qa in paragraph.get("qas", []):
                rows.append(_row({"id":qa.get("id"), "context":context,
                                  "question":qa.get("question"),
                                  "answers":{"text":[a.get("text", "") for a in qa.get("answers", [])]}}))
    return rows

def _load_jsonl_qa(path):
    """Load the project JSONL schema: id, question, context, answers."""
    rows=[]; seen=set()
    with open(path, encoding="utf-8") as stream:
        for line in stream:
            if not line.strip(): continue
            item=json.loads(line)
            row={"id":str(item.get("id", "")), "question":normalize(item.get("question")),
                 "context":normalize(item.get("context")),
                 "answers":[normalize(x) for x in (item.get("answers") or []) if normalize(x)],
                 "source":str(item.get("source", "external jsonl")),
                 "wiki_id":str(item.get("wiki_id", ""))}
            if not row["question"] or not row["context"] or not row["answers"]: continue
            key=hashlib.sha256((row["question"]+"\0"+row["context"]+"\0"+"|".join(row["answers"])).encode()).hexdigest()
            if key in seen: continue
            seen.add(key); rows.append(row)
    return rows

def _looks_like_hf_dataset(source):
    text = str(source)
    if text.endswith(".jsonl") or text.startswith((".", "/", "~")):
        return False
    parts = text.split("/")
    return len(parts) == 2 and all(parts)


def _resolve_external_jsonl(source, split, cache_dir=None):
    """Resolve a local JSONL path or a Hugging Face dataset repo to a local file.

    Accepts ``data/indic_rag_te/train.jsonl`` or a dataset repo id such as
    ``nlpctx/activeqa-telugu-rl-data``. For a repo id, the requested split file
    (``train.jsonl`` or ``dev.jsonl``) is downloaded from the Hub.
    """
    path = Path(str(source)).expanduser()
    if path.is_file():
        return str(path)
    if not _looks_like_hf_dataset(source):
        raise FileNotFoundError(f"External QA source not found: {source}")
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=str(source), filename=f"{split}.jsonl",
                           repo_type="dataset", cache_dir=cache_dir)


def _resolve_dataset_file(source, filename, fallback, cache_dir=None):
    """Resolve a nested file in a dataset repo, preferring an existing local copy."""
    path = Path(str(fallback)).expanduser() if fallback else None
    if path is not None and path.is_file():
        return str(path)
    if not source or not _looks_like_hf_dataset(source):
        raise FileNotFoundError(f"Dataset file not found: {fallback or filename}")
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=str(source), filename=filename,
                           repo_type="dataset", cache_dir=cache_dir)


def _load_reformulation_jsonl(path):
    """Read the uploaded rule_reformulations split as question/target pairs."""
    rows = []
    with open(path, encoding="utf-8") as stream:
        for line in stream:
            if not line.strip():
                continue
            item = json.loads(line)
            question = normalize(item.get("question"))
            target = normalize(item.get("reformulation") or item.get("candidate"))
            if not question or not target or question == target:
                continue
            rows.append({
                "id": str(item.get("id", len(rows))),
                "question": question,
                "reformulation": target,
                "context": normalize(item.get("context")),
                "answers": [normalize(x) for x in (item.get("answers") or []) if normalize(x)],
                "wiki_id": str(item.get("wiki_id", "")),
                "label_status": str(item.get("label_status", "silver_rule_based_requires_telugu_review")),
            })
    return rows


def load_reformulation_splits(cfg):
    """Load distinct train/dev rewrite pairs from the configured HF dataset repo."""
    repo = cfg.get("reformulation_dataset") or cfg.get("external_train_dataset")
    cache_dir = cfg.get("cache_dir")
    train_file = _resolve_dataset_file(
        repo, "rule_reformulations/train.jsonl",
        cfg.get("reformulation_train_file", "data/indic_rag_te/rule_reformulations/train.jsonl"),
        cache_dir,
    )
    dev_file = _resolve_dataset_file(
        repo, "rule_reformulations/dev.jsonl",
        cfg.get("reformulation_dev_file", "data/indic_rag_te/rule_reformulations/dev.jsonl"),
        cache_dir,
    )
    train = _load_reformulation_jsonl(train_file)
    dev = _load_reformulation_jsonl(dev_file)
    if cfg.get("reformulation_train_examples") is not None:
        train = train[:int(cfg["reformulation_train_examples"])]
    if cfg.get("reformulation_dev_examples") is not None:
        dev = dev[:int(cfg["reformulation_dev_examples"])]
    train_articles = {r["wiki_id"] for r in train if r["wiki_id"]}
    dev_articles = {r["wiki_id"] for r in dev if r["wiki_id"]}
    overlap = train_articles & dev_articles
    if overlap:
        raise ValueError(f"Reformulation train/dev article leakage: {len(overlap)} wiki_id values")
    print(f"reformulation_train_source={train_file} rows={len(train)}; "
          f"reformulation_dev_source={dev_file} rows={len(dev)}; article_overlap=0")
    return train, dev


def load_splits(cfg, include_full_train=False):
    """Load external Telugu training rows and a disjoint IndicQA test evaluation.

    The upstream repository provides a legacy custom loading script, which Datasets 5.x
    intentionally no longer executes. Fetch only the requested language JSON and parse
    the documented SQuAD-style structure directly. Without ``external_train_file``, retain
    the old exploratory split of IndicQA test rows for backwards compatibility.
    """
    from huggingface_hub import hf_hub_download
    dataset=cfg.get("dataset", "ai4bharat/IndicQA")
    language=cfg.get("config", "indicqa.te").split(".")[-1]
    if language != "te":
        raise ValueError(f"This implementation currently expects Telugu config indicqa.te, got {language!r}")
    cache_dir=cfg.get("cache_dir")
    filepath=hf_hub_download(repo_id=dataset, filename=f"data/indicqa.{language}.json",
                             repo_type="dataset", cache_dir=cache_dir)
    rows=[]; seen=set()
    for x in _load_indicqa_json(filepath):
        if not x["question"] or not x["context"] or not x["answers"]: continue
        key=hashlib.sha256((x["question"]+"\0"+x["context"]+"\0"+"|".join(x["answers"])).encode()).hexdigest()
        if key in seen: continue
        seen.add(key); rows.append(x)
    import random
    rng=random.Random(42); rng.shuffle(rows)
    external_train=cfg.get("external_train_dataset") or cfg.get("external_train_file")
    if external_train:
        try:
            train_source=_resolve_external_jsonl(external_train, "train", cache_dir)
        except Exception as exc:
            fallback=cfg.get("external_train_file")
            if not fallback or fallback==external_train:
                raise
            print(f"external_dataset_unavailable={external_train} "
                  f"({type(exc).__name__}: {exc}); falling back to {fallback}")
            train_source=_resolve_external_jsonl(fallback, "train", cache_dir)
        print(f"external_train_source={train_source}")
        full_train=_load_jsonl_qa(train_source)
        ev=rows
        if cfg.get("eval_examples") is not None: ev=ev[:int(cfg["eval_examples"])]
        heldout_contexts={r["context"].casefold() for r in ev}
        before=len(full_train)
        full_train=[r for r in full_train if r["context"].casefold() not in heldout_contexts]
        print(f"external_train_rows={before}; dropped_exact_eval_context_duplicates={before-len(full_train)}")
        tr=full_train
        if cfg.get("train_examples") is not None: tr=tr[:int(cfg["train_examples"])]
        if include_full_train: return tr,ev,full_train
        return tr,ev
    n_eval=max(1, int(len(rows)*cfg.get("eval_fraction", .2)))
    ev=rows[:n_eval]; full_train=rows[n_eval:]; tr=full_train
    if cfg.get("train_examples") is not None: tr=tr[:int(cfg["train_examples"])]
    if cfg.get("eval_examples") is not None: ev=ev[:int(cfg["eval_examples"])]
    if include_full_train:
        return tr,ev,full_train
    return tr,ev

def save_splits(train, evaluation, out="outputs/split_manifest.json"):
    Path(out).parent.mkdir(parents=True,exist_ok=True)
    Path(out).write_text(json.dumps({"seed":42,"train_ids":[x["id"] for x in train],"eval_ids":[x["id"] for x in evaluation],"train_size":len(train),"eval_size":len(evaluation),"train_source":train[0].get("source","IndicQA exploratory") if train else None,"evaluation_source":"ai4bharat/IndicQA test"},ensure_ascii=False,indent=2),encoding="utf-8")

In [ ]:
%%writefile /kaggle/working/activeqa/src/activeqa/qa_model.py
"""Frozen CPU extractive QA with context-only span decoding."""
import inspect
import time

import torch


def _context_token_mask(encoded, inputs, batch_index, device):
    if "token_type_ids" in encoded:
        return encoded["token_type_ids"][batch_index].eq(1).to(device)
    try:
        sequence_ids = encoded.sequence_ids(batch_index)
    except (AttributeError, TypeError):
        sequence_ids = None
    if sequence_ids is None:
        return None
    return torch.tensor([sid == 1 for sid in sequence_ids], device=device)


class TeluguQA:
    def __init__(self, name, cache_dir="models", max_length=384, stride=64, device="cpu"):
        from transformers import AutoTokenizer, AutoModelForQuestionAnswering
        self.tok = AutoTokenizer.from_pretrained(name, cache_dir=cache_dir)
        self.device = torch.device(device)
        self.model = AutoModelForQuestionAnswering.from_pretrained(
            name, cache_dir=cache_dir).to(self.device).eval()
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)
        self.max_length = max_length
        self.stride = min(stride, max(0, max_length // 4))
        self.forward_fields = set(inspect.signature(self.model.forward).parameters)

    @torch.inference_mode()
    def answer(self, question, context):
        text, elapsed, _ = self.answer_with_score(question, context)
        return text, elapsed

    @torch.inference_mode()
    def answer_with_score(self, question, context):
        """Return answer text, elapsed seconds, and a context-span confidence score.

        Confidence is the best start/end log-probability sum among legal context
        spans. It is an inference-time ranking signal, not a calibrated probability.
        """
        started = time.perf_counter()
        encoded = self.tok(
            question, context, return_tensors="pt", truncation="only_second",
            max_length=self.max_length, stride=self.stride,
            return_overflowing_tokens=True, return_offsets_mapping=True,
            padding=True,
        )
        offsets = encoded.pop("offset_mapping")
        encoded.pop("overflow_to_sample_mapping", None)
        inputs = {key: value.to(self.device) for key, value in encoded.items()
                  if key in self.forward_fields and key in {
                      "input_ids", "attention_mask", "token_type_ids"}}
        output = self.model(**inputs)
        best = (float("-inf"), "")
        for batch_index in range(inputs["input_ids"].shape[0]):
            start_logits = output.start_logits[batch_index].clone()
            end_logits = output.end_logits[batch_index].clone()
            context_mask = _context_token_mask(encoded, inputs, batch_index, self.device)
            if context_mask is None:
                continue
            context_mask &= inputs["attention_mask"][batch_index].bool()
            if not context_mask.any():
                continue
            start_logits[~context_mask] = -1e9
            end_logits[~context_mask] = -1e9
            start_logp = torch.log_softmax(start_logits, dim=-1)
            end_logp = torch.log_softmax(end_logits, dim=-1)
            starts = torch.topk(start_logp, k=min(20, len(start_logp))).indices.tolist()
            ends = torch.topk(end_logp, k=min(20, len(end_logp))).indices.tolist()
            for start in starts:
                for end in ends:
                    if end < start or end - start > 30:
                        continue
                    score = float(start_logp[start] + end_logp[end])
                    if score > best[0]:
                        char_start = int(offsets[batch_index, start, 0])
                        char_end = int(offsets[batch_index, end, 1])
                        best = (score, context[char_start:char_end].strip())
        return best[1], time.perf_counter() - started, best[0]

    @torch.inference_mode()
    def answer_batch(self, questions, context):
        """Answer several questions against one shared context in one forward pass.

        Returns ``(answer, elapsed_seconds, confidence)`` per question, in input
        order. This is used by training to score the original question and all
        sampled rewrites together instead of one QA call each.
        """
        questions = list(questions)
        if not questions:
            return []
        started = time.perf_counter()
        encoded = self.tok(
            questions, [context] * len(questions), return_tensors="pt",
            truncation="only_second",
            max_length=self.max_length, stride=self.stride,
            return_overflowing_tokens=True, return_offsets_mapping=True,
            padding=True,
        )
        offsets = encoded.pop("offset_mapping")
        sample_map = encoded.pop("overflow_to_sample_mapping").tolist()
        inputs = {key: value.to(self.device) for key, value in encoded.items()
                  if key in self.forward_fields and key in {
                      "input_ids", "attention_mask", "token_type_ids"}}
        output = self.model(**inputs)
        best = [(float("-inf"), "") for _ in questions]
        for batch_index in range(inputs["input_ids"].shape[0]):
            sample_index = sample_map[batch_index]
            start_logits = output.start_logits[batch_index].clone()
            end_logits = output.end_logits[batch_index].clone()
            context_mask = _context_token_mask(encoded, inputs, batch_index, self.device)
            if context_mask is None:
                continue
            context_mask &= inputs["attention_mask"][batch_index].bool()
            if not context_mask.any():
                continue
            start_logits[~context_mask] = -1e9
            end_logits[~context_mask] = -1e9
            start_logp = torch.log_softmax(start_logits, dim=-1)
            end_logp = torch.log_softmax(end_logits, dim=-1)
            starts = torch.topk(start_logp, k=min(20, len(start_logp))).indices.tolist()
            ends = torch.topk(end_logp, k=min(20, len(end_logp))).indices.tolist()
            for start in starts:
                for end in ends:
                    if end < start or end - start > 30:
                        continue
                    score = float(start_logp[start] + end_logp[end])
                    if score > best[sample_index][0]:
                        char_start = int(offsets[batch_index, start, 0])
                        char_end = int(offsets[batch_index, end, 1])
                        best[sample_index] = (score, context[char_start:char_end].strip())
        elapsed = time.perf_counter() - started
        return [(answer, elapsed, score) for score, answer in best]

In [ ]:
%%writefile /kaggle/working/activeqa/src/activeqa/rewards.py
from .preprocessing import tokens

def token_f1(prediction, references):
    p=tokens(prediction); best=0.0
    for ref in references or [""]:
        g=tokens(ref)
        if not p and not g: score=1.
        elif not p or not g: score=0.
        else:
            from collections import Counter
            overlap=sum((Counter(p)&Counter(g)).values()); precision=overlap/len(p); recall=overlap/len(g)
            score=2*precision*recall/(precision+recall) if precision+recall else 0.
        best=max(best,score)
    return best

def reward(prediction,references,valid=True,preserve=1.0,em_weight=.1,preserve_weight=.1,invalid_penalty=.2):
    f1=token_f1(prediction,references); em=float(any(" ".join(tokens(prediction))==" ".join(tokens(x)) for x in references))
    return f1+em_weight*em+preserve_weight*preserve-(0 if valid else invalid_penalty)

In [ ]:
%%writefile /kaggle/working/activeqa/src/activeqa/policy_gradient.py
from __future__ import annotations


class RunningBaseline:
    def __init__(self,momentum=.9): self.momentum=momentum; self.value=None
    def advantage(self,reward):
        old=self.value if self.value is not None else 0.0
        adv=reward-old
        self.value=reward if self.value is None else self.momentum*self.value+(1-self.momentum)*reward
        return adv

def reinforce_step(logprob,reward_value,baseline,optimizer,params,clip=1.0):
    import torch
    adv=baseline.advantage(float(reward_value))
    loss=-logprob*torch.tensor(adv,dtype=logprob.dtype)
    if not torch.isfinite(loss): raise FloatingPointError("non-finite policy loss")
    optimizer.zero_grad(set_to_none=True); loss.backward()
    grads=[p.grad for p in params if p.grad is not None]
    if not grads: raise RuntimeError("REINFORCE produced no trainable gradients")
    if not all(torch.isfinite(g).all() for g in grads): raise FloatingPointError("non-finite policy gradient")
    torch.nn.utils.clip_grad_norm_(params,clip)
    optimizer.step()
    return float(loss.detach()),adv


def sft_batch_step(logprobs, optimizer, params, clip=1.0):
    """Teacher-forcing update on selected high-reward sequences.

    Used by rejection-sampling fine-tuning: only sequences that were sampled,
    validated, and scored at least as well as the original question are passed
    here. The loss increases their likelihood, so a batch with no good sample
    is skipped instead of pushing every sampled action down as REINFORCE would.
    """
    import torch
    if not logprobs:
        raise ValueError("At least one target sequence is required")
    loss = -torch.stack(logprobs).mean()
    if not torch.isfinite(loss):
        raise FloatingPointError("non-finite SFT loss")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grads = [p.grad for p in params if p.grad is not None]
    if not grads:
        raise RuntimeError("SFT produced no trainable gradients")
    if not all(torch.isfinite(g).all() for g in grads):
        raise FloatingPointError("non-finite SFT gradient")
    torch.nn.utils.clip_grad_norm_(params, clip)
    optimizer.step()
    return float(loss.detach())


def reinforce_batch_step(logprobs, advantages, optimizer, params, clip=1.0):
    """One REINFORCE update over a batch of sampled actions.

    Advantages are expected to be per-question centred (typically against the
    original-question reward). Scaling by their batch standard deviation keeps
    the update magnitude stable without removing the sign of the signal.
    """
    import torch
    if not logprobs:
        raise ValueError("At least one sampled action is required")
    if len(logprobs) != len(advantages):
        raise ValueError("logprobs and advantages must have the same length")
    device = logprobs[0].device
    adv = torch.as_tensor(advantages, dtype=logprobs[0].dtype, device=device)
    if adv.numel() > 1:
        adv = adv / (adv.std(unbiased=False) + 1e-6)
    loss = -(torch.stack(logprobs) * adv).sum() / len(logprobs)
    if not torch.isfinite(loss):
        raise FloatingPointError("non-finite policy loss")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grads = [p.grad for p in params if p.grad is not None]
    if not grads:
        raise RuntimeError("REINFORCE produced no trainable gradients")
    if not all(torch.isfinite(g).all() for g in grads):
        raise FloatingPointError("non-finite policy gradient")
    torch.nn.utils.clip_grad_norm_(params, clip)
    optimizer.step()
    return float(loss.detach())

In [ ]:
%%writefile /kaggle/working/activeqa/src/activeqa/rft.py
"""On-policy rejection-sampling fine-tuning (RFT) for Telugu reformulation.

The SFT adapter is a good paraphraser, but it was trained to imitate reference
paraphrases, not to make a downstream QA model answer correctly. This stage
closes that gap with the frozen Telugu QA model as the reward:

1. Sample ``candidates`` rewrites per question from the current policy.
2. Answer the original question and every rewrite on the same passage.
3. Keep rewrites that are valid, score at least as well as the original
   (``margin``), and whose answer span is at least as confident as the
   original's (``confidence_slack``, default 0.0 because the deployed selector
   only switches on ``confidence > original``). Training on rewrites the
   selector would never pick does not change held-out QA.
4. Teacher-force the best qualifying rewrite, accumulating until
   ``batch_questions`` targets, then take one optimizer step.

Unlike REINFORCE, a batch with no qualifying rewrite is skipped rather than
pushing every sampled action down, which is what made the earlier all-negative
REINFORCE runs flat or harmful.
"""
from __future__ import annotations

import math
import random
import time

import torch

from .preprocessing import entity_signature
from .policy_gradient import sft_batch_step
from .rewards import reward, token_f1


def _chunks(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]


def rft_on_policy(ref, qa, rows, *, candidates=6, batch_questions=8,
                  epochs=1, learning_rate=7e-5, margin=0.0,
                  confidence_slack=0.0, max_input=96, max_new=64,
                  do_sample=True, seed=42, generate_batch=8,
                  max_updates=None, after_update=None, skip_solved=True,
                  preserve_weight=None, log=print):
    """Run RFT on ``rows`` (dicts with question/context/answers).

    ``skip_solved``: questions whose original answer is already correct
    (token-F1 ≥ 0.99) are skipped — there is no headroom to learn from and
    training on them only teaches safe paraphrasing of solved cases.
    ``preserve_weight`` overrides the reward's entity-preservation weight
    (the checkpoint's default is 0.1); raising it makes the reward punish
    rewrites that drift entity text (the observed పెద్దగుమ్మడం →
    పెద్దబొమ్మడం failure mode) instead of mildly discounting them.
    ``after_update(n)`` is called after every optimizer step that reaches
    ``n`` updates; the pipeline uses it to snapshot + evaluate adapters so a
    late drift can never destroy an earlier improvement (checkpoint selection).
    """
    if not rows:
        raise ValueError("RFT needs at least one training question")
    reward_kwargs = {"preserve_weight": preserve_weight} if preserve_weight is not None else {}
    trainable = [p for p in ref.model.parameters() if p.requires_grad]
    if not trainable:
        raise RuntimeError("The reformulator has no trainable parameters; "
                           "did the LoRA adapter load?")
    optimizer = torch.optim.AdamW(trainable, lr=float(learning_rate),
                                  weight_decay=0.0)

    log(f"      RFT: {len(rows)} questions × {epochs} epoch(s), "
        f"{candidates} candidates, {batch_questions} targets/update, "
        f"lr={learning_rate}, margin={margin}, conf_slack={confidence_slack}, "
        f"skip_solved={skip_solved}")
    log(f"      RFT trainable params: {sum(p.numel() for p in trainable):,}")

    pending = []          # [(question, token_ids)]
    losses = []
    records = []
    scanned = scored = valid_actions = 0
    selected_questions = 0
    solved_skipped = 0
    question_errors = 0
    update_skipped = 0
    original_reward_sum = 0.0
    selected_reward_sum = 0.0
    selected_original_reward_sum = 0.0
    candidate_reward_sum = 0.0
    candidate_count = 0
    started = time.perf_counter()
    ref.model.train()

    def flush():
        nonlocal update_skipped
        if not pending:
            return None
        try:
            logprobs = ref.sequence_logprob_batch(pending, max_input)
            loss = sft_batch_step(list(logprobs), optimizer, trainable, clip=1.0)
        except (FloatingPointError, RuntimeError) as exc:
            update_skipped += 1
            optimizer.zero_grad(set_to_none=True)
            log(f"      RFT skipped a non-finite update ({type(exc).__name__}: {exc})")
            pending.clear()
            return None
        losses.append(loss)
        pending.clear()
        if after_update is not None:
            ref.model.train()
            after_update(len(losses))
        return loss

    for epoch in range(epochs):
        order = list(range(len(rows)))
        random.Random(seed + epoch).shuffle(order)
        log(f"      RFT epoch {epoch + 1}/{epochs} …")
        for chunk in _chunks(order, max(1, generate_batch)):
            questions = [rows[i]["question"] for i in chunk]
            try:
                actions_per_question = ref.sample_actions_batch(
                    questions, candidates, max_input, max_new, do_sample)
            except RuntimeError as exc:
                question_errors += len(chunk)
                log(f"      RFT generation failed for {len(chunk)} questions: {exc}")
                continue

            for row, question, actions in zip((rows[i] for i in chunk),
                                              questions, actions_per_question):
                scanned += 1
                try:
                    answers = qa.answer_batch(
                        [question] + [action["candidate"] for action in actions],
                        row["context"])
                except RuntimeError as exc:
                    question_errors += 1
                    log(f"      RFT QA scoring failed ({type(exc).__name__}): {exc}")
                    continue
                original_reward = reward(answers[0][0], row["answers"],
                                         valid=True, preserve=1.0, **reward_kwargs)
                original_confidence = answers[0][2]
                original_reward_sum += original_reward
                if (skip_solved and margin >= 0.0
                        and token_f1(answers[0][0], row["answers"]) >= 0.99):
                    solved_skipped += 1
                    continue

                values = []
                confidences = []
                for action, (answer, _, confidence) in zip(actions, answers[1:]):
                    preserve = entity_signature(question).issubset(
                        entity_signature(action["sampled_text"]))
                    values.append(reward(answer, row["answers"],
                                         valid=action["valid"],
                                         preserve=float(preserve), **reward_kwargs))
                    confidences.append(confidence)
                    valid_actions += int(action["valid"])
                    candidate_reward_sum += values[-1]
                    candidate_count += 1
                scored += len(actions)

                eligible = [i for i in range(len(actions))
                            if actions[i]["valid"]
                            and values[i] + 1e-9 >= original_reward + margin
                            and confidences[i] >= original_confidence - confidence_slack]
                if not eligible:
                    continue
                best = max(eligible, key=lambda i: (values[i], confidences[i]))
                selected_questions += 1
                selected_reward_sum += values[best]
                selected_original_reward_sum += original_reward
                pending.append((question, actions[best]["token_ids"]))
                if len(records) < 2000:
                    records.append({
                        "epoch": epoch + 1,
                        "question": question,
                        "original_reward": original_reward,
                        "original_confidence": original_confidence,
                        "selected_reward": values[best],
                        "selected_confidence": confidences[best],
                        "candidate_rewards": values,
                        "candidate_validity": [a["valid"] for a in actions],
                        "selected_text": actions[best]["sampled_text"],
                        "selected_valid": actions[best]["valid"],
                        "reward_lift": values[best] - original_reward,
                    })
                if len(pending) >= batch_questions:
                    flush()
                    if max_updates is not None and len(losses) >= max_updates:
                        break

            if max_updates is not None and len(losses) >= max_updates:
                break
            if scanned and scanned % 24 == 0:
                recent = records[-24:]
                elapsed = time.perf_counter() - started
                rate = scanned / max(1e-9, elapsed)
                log(f"      RFT {scanned}/{len(rows) * epochs} scanned  "
                    f"updates={len(losses)}  "
                    f"valid={valid_actions / max(1, scored):.2f}  "
                    f"selected={selected_questions / scanned:.2f}  "
                    f"lift={sum(x['reward_lift'] for x in recent) / max(1, len(recent)):+.3f}  "
                    f"loss={(f'{sum(losses[-20:]) / len(losses[-20:]):.3f}' if losses else '—')}  "
                    f"{rate:.2f} q/s")
        if max_updates is not None and len(losses) >= max_updates:
            if pending:
                flush()
            break
    if pending:
        flush()

    elapsed = time.perf_counter() - started
    if question_errors and scanned and question_errors / max(1, scanned) > 0.10:
        raise RuntimeError(f"RFT could not score {question_errors}/{scanned} questions; "
                           "aborting instead of training on a broken loop")
    metrics = {
        "questions_seen": scanned,
        "candidate_actions": scored,
        "optimizer_steps": len(losses),
        "skipped_updates": update_skipped,
        "solved_skipped": solved_skipped,
        "solved_skip_rate": round(solved_skipped / max(1, scanned), 4),
        "question_errors": question_errors,
        "valid_generation_rate": round(valid_actions / max(1, scored), 4),
        "selected_rate": round(selected_questions / max(1, scanned), 4),
        "mean_original_reward": round(original_reward_sum / max(1, scanned), 4),
        "mean_candidate_reward": round(candidate_reward_sum / max(1, candidate_count), 4),
        "mean_selected_reward": round(selected_reward_sum / max(1, selected_questions), 4),
        "mean_reward_lift": round((selected_reward_sum - selected_original_reward_sum)
                                  / max(1, selected_questions), 4),
        "initial_loss": round(losses[0], 4) if losses else None,
        "final_loss": round(sum(losses[-20:]) / len(losses[-20:]), 4) if losses else None,
        "elapsed_seconds": round(elapsed, 1),
        "questions_per_second": round(scanned / max(1e-9, elapsed), 2),
        "settings": {
            "candidates_per_question": candidates,
            "batch_questions": batch_questions,
            "epochs": epochs,
            "learning_rate": learning_rate,
            "min_reward_margin": margin,
            "confidence_slack": confidence_slack,
            "max_input_tokens": max_input,
            "max_new_tokens": max_new,
            "do_sample": do_sample,
            "seed": seed,
            "skip_solved_originals": skip_solved,
            "preserve_weight": preserve_weight if preserve_weight is not None else 0.1,
        },
    }
    log(f"      RFT done: {scanned} questions, {len(losses)} updates, "
        f"selected={metrics['selected_rate']:.2f}, "
        f"lift={metrics['mean_reward_lift']:+.3f}, "
        f"{elapsed / 60:.1f} min")
    return metrics, records

In [ ]:
%%writefile /kaggle/working/activeqa/src/activeqa/reformulator.py
from __future__ import annotations
import os
import re
import torch
from .preprocessing import normalize, entity_signature, interrogatives, tokens


_TELUGU_RE = re.compile(r"[\u0c00-\u0c7f]")
_FOREIGN_INDIC_RE = re.compile(r"[\u0900-\u0bff\u0c80-\u0d7f]")
_OUTPUT_MARKER_RE = re.compile(r"\[(?:more|continued|truncated)\]", re.I)
_TELUGU_FUNCTION_WORDS = {
    "ఏ", "ఏది", "ఏమి", "ఏమిటి", "ఎవరు", "ఎప్పుడు", "ఎక్కడ", "ఎందుకు", "ఎలా",
    "ఎన్ని", "ఎంత", "లో", "కి", "కు", "తో", "మరియు", "అని", "గా", "నే", "ను",
    "ది", "లు", "ఒక", "ఈ", "ఆ", "ఏదైనా", "చెప్పండి", "తెలుపుము", "పేర్కొనండి",
}


def _answer_type(text):
    words = set(tokens(text))
    if "శతాబ్దం" in words or "శతాబ్దంలో" in words or "శతాబ్దానికి" in words:
        return "century"
    if any(w.startswith("సంవత్సర") for w in words): return "year"
    if any(w.startswith("తేదీ") for w in words): return "date"
    if any(w.startswith("ఎన్ని") for w in words) or "సంఖ్య" in words: return "count"
    if "ఎంత" in words: return "amount"
    if any(w in words for w in ("ఎక్కడ", "ప్రదేశం", "నగరంలో", "స్థానం")): return "place"
    if "ఎందుకు" in words: return "reason"
    if any(w in words for w in ("ఎలా", "విధంగా")): return "manner"
    if any(w in words for w in ("ఎప్పుడు", "సమయంలో", "కాలంలో")): return "time"
    if any(w in words for w in ("ఎవరు", "ఎవరిని", "ఎవరి", "వ్యక్తి", "రచయిత", "ప్రధాని", "రాష్ట్రపతి")) or any(w.startswith("కళాకార") for w in words):
        return "person"
    if any(w in words for w in ("ఏది", "ఏమిటి", "ఏమి", "పేరు", "ఏ")): return "thing"
    return "unknown"


def _preserves_answer_type(source, candidate):
    src, dst = _answer_type(source), _answer_type(candidate)
    if src == dst:
        return True
    if src == "unknown" or dst == "unknown":
        return True
    if src == "person" and dst == "thing":
        return bool(re.search(r"పేరు|వ్యక్తి|రచయిత|కళాకార", candidate))
    return False


def _anchor_overlap(source, candidate):
    """Conservative lexical guard against dropping Telugu names/topic words.

    Only words of four or more characters are considered anchors (short Telugu
    words are usually particles), and a rewrite needs at least one shared
    anchor plus 25% of them. The earlier 55% rule rejected most of the SFT
    model's genuine paraphrases — it treated every content word as a proper
    name, even though Telugu paraphrases routinely replace verbs and adverbs.
    """
    source_words = [w for w in tokens(source) if len(w) >= 4 and w not in _TELUGU_FUNCTION_WORDS]
    target_words = tokens(candidate)
    if not source_words:
        return True
    from difflib import SequenceMatcher
    matched = 0
    for word in source_words:
        if any(word == other or word in other or other in word or
               SequenceMatcher(None, word, other).ratio() >= .70
               for other in target_words):
            matched += 1
    return matched >= 1 and matched / len(source_words) >= .20

class Reformulator:
    def __init__(self, name, cache_dir="models", rank=None, alpha=None, language="te",
                 device="cpu", adapter_path=None, include_mlp=False, prompt=None):
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        try:
            self.tokenizer=AutoTokenizer.from_pretrained(
                name, cache_dir=cache_dir, use_fast=False,
                do_lower_case=False, keep_accents=True,
            )
        except (TypeError, ValueError):
            self.tokenizer=AutoTokenizer.from_pretrained(
                name, cache_dir=cache_dir, use_fast=False)
        self.device=torch.device(device)
        self.model=AutoModelForSeq2SeqLM.from_pretrained(name,cache_dir=cache_dir)
        self.model.train()
        self.base_name=name
        self.language=language
        self.adapter_path=adapter_path
        self.target_lang_token=f"<2{language}>"
        self.target_lang_id=self.tokenizer._convert_token_to_id_with_added_voc(self.target_lang_token)
        if self.target_lang_id == self.tokenizer.unk_token_id:
            self.prompt_template=(prompt
                or "Rephrase the question below in the same language.\n"
                   "Output ONLY the rephrased question — do not answer it.\n"
                   "{question}")
            self.target_lang_id=None
            self.bos_token_id=self.tokenizer.bos_token_id
            self.eos_token_id=(self.tokenizer.eos_token_id
                               if self.tokenizer.eos_token_id is not None else None)
        else:
            self.prompt_template=None
            self.bos_token_id=self.tokenizer._convert_token_to_id_with_added_voc("<s>")
            self.eos_token_id=self.tokenizer._convert_token_to_id_with_added_voc("</s>")
            if self.bos_token_id == self.tokenizer.unk_token_id:
                raise RuntimeError("Tokenizer does not define the expected <s> start token")
            if self.eos_token_id == self.tokenizer.unk_token_id:
                self.eos_token_id=self.tokenizer.eos_token_id
            if self.eos_token_id is None or self.eos_token_id == self.tokenizer.unk_token_id:
                raise RuntimeError("Tokenizer does not define the expected </s> end token")
            self.model.config.decoder_start_token_id=self.target_lang_id
            self.model.generation_config.decoder_start_token_id=self.target_lang_id
            self.model.config.bos_token_id=self.bos_token_id
            self.model.config.eos_token_id=self.eos_token_id
            self.model.generation_config.bos_token_id=self.bos_token_id
            self.model.generation_config.eos_token_id=self.eos_token_id
        if adapter_path:
            if not os.path.isdir(adapter_path):
                raise FileNotFoundError(f"Adapter directory not found: {adapter_path}")
            from peft import PeftModel
            self.model = PeftModel.from_pretrained(
                self.model, adapter_path, is_trainable=True)
            self.model.print_trainable_parameters()
        elif rank is not None:
            from peft import LoraConfig, TaskType, get_peft_model
            targets = ["q_proj", "k_proj", "v_proj", "out_proj", "q", "k", "v", "o"]
            if include_mlp:
                targets += ["fc1", "fc2", "wi", "wo", "wi_0", "wi_1"]
            self.model = get_peft_model(self.model, LoraConfig(
                r=rank, lora_alpha=(alpha or 2 * rank),
                lora_dropout=0.05, bias="none", task_type=TaskType.SEQ_2_SEQ_LM,
                target_modules=targets))
            self.model.print_trainable_parameters()
        self.model.to(self.device)
        from transformers import PreTrainedModel
        language_model = self.model
        while not isinstance(language_model, PreTrainedModel):
            language_model = language_model.get_base_model()
        encoder_embedding = language_model.get_input_embeddings().weight
        decoder_embedding = language_model.get_decoder().embed_tokens.weight
        if (decoder_embedding is not encoder_embedding
                and not torch.equal(decoder_embedding, encoder_embedding)):
            raise RuntimeError(
                "Encoder/decoder embeddings are not tied to the shared table. "
                "Install transformers==4.57.6 (the checkpoint was trained with "
                "tied embeddings) before loading this reformulator.")
        self.cache={}
        if adapter_path is None:
            if hasattr(self.model, "gradient_checkpointing_enable"):
                self.model.gradient_checkpointing_enable()
            if hasattr(self.model.config, "use_cache"):
                self.model.config.use_cache = False
        self.trainable=[p for p in self.model.parameters() if p.requires_grad]
        self.trainable_count=sum(p.numel() for p in self.trainable)

    def _build_text(self, question):
        q = normalize(question)
        if self.prompt_template is not None:
            return self.prompt_template.format(question=q)
        return f"{q} </s> {self.target_lang_token}"

    def _source(self, question, max_input):
        text=self._build_text(question)
        encoded=self.tokenizer(text,add_special_tokens=self.prompt_template is not None,
                               return_tensors="pt",truncation=True,max_length=max_input)
        return {key:value.to(self.device) for key,value in encoded.items() if key in {"input_ids","attention_mask"}}

    def sample(self, question, n=2, max_input=96, max_new=64, do_sample=True):
        """Return display candidates, replacing invalid outputs with the input question."""
        return [x["candidate"] for x in self.sample_actions(question, n, max_input, max_new, do_sample)]

    def _decode_actions(self, question, sequences):
        actions=[]
        for ids in sequences:
            text=normalize(self.tokenizer.decode(ids,skip_special_tokens=True,clean_up_tokenization_spaces=False))
            tokens=ids.tolist() if hasattr(ids,"tolist") else list(ids)
            terminated = self.eos_token_id in tokens
            if tokens and tokens[0] == self.target_lang_id:
                tokens=tokens[1:]
            if self.eos_token_id in tokens:
                tokens=tokens[:tokens.index(self.eos_token_id)+1]
            else:
                tokens=[x for x in tokens if x != self.tokenizer.pad_token_id]
            question_type_preserved=_preserves_answer_type(question, text)
            valid=self._valid(question,text,terminated=terminated)
            actions.append({"sampled_text":text,"candidate":text if valid else question,
                            "valid":valid,"terminated":terminated,"token_ids":tokens,
                            "question_type_preserved":question_type_preserved})
        return actions

    def sample_actions(self, question, n=2, max_input=96, max_new=64, do_sample=True):
        """Sample policy actions while retaining the exact generated token IDs.

        Invalid text is exposed as the original question for downstream answer scoring,
        but the original sampled IDs remain attached for the policy gradient. This keeps
        the REINFORCE likelihood paired with the actual sampled action.
        """
        return self.sample_actions_batch([question], n, max_input, max_new, do_sample)[0]

    def sample_actions_batch(self, questions, n=2, max_input=96, max_new=64, do_sample=True):
        """Sample ``n`` actions for several questions in one generate() call.

        Batching is the main speedup for on-policy RL: eight questions × six
        candidates cost one encoder pass instead of eight, and the decoder still
        produces the same number of sequences. Per-question validity/gating is
        applied after decoding, exactly as in :meth:`sample_actions`.
        """
        questions=[normalize(q) for q in questions]
        if not questions:
            return []
        encoded=self.tokenizer(
            [self._build_text(q) for q in questions],
            add_special_tokens=self.prompt_template is not None, return_tensors="pt",
            padding=True, truncation=True, max_length=max_input)
        inp={key:value.to(self.device) for key,value in encoded.items()
             if key in {"input_ids","attention_mask"}}
        was_training=self.model.training
        self.model.eval()
        generation_kwargs={"max_new_tokens":max_new,"do_sample":do_sample,
                           "num_beams":max(1,n) if not do_sample else 1,
                           "num_return_sequences":n,
                           "pad_token_id":self.tokenizer.pad_token_id
                               if self.tokenizer.pad_token_id is not None
                               else self.eos_token_id,
                           "use_cache":True}
        if self.target_lang_id is not None:
            generation_kwargs.update(bos_token_id=self.bos_token_id,
                                     eos_token_id=self.eos_token_id,
                                     decoder_start_token_id=self.target_lang_id)
        generation_kwargs.update(no_repeat_ngram_size=3, encoder_no_repeat_ngram_size=3)
        if do_sample:
            generation_kwargs.update(top_p=.9,temperature=.9)
        try:
            with torch.inference_mode():
                seq=self.model.generate(**inp,**generation_kwargs)
        finally:
            self.model.train(was_training)
        grouped=seq.reshape(len(questions), n, -1)
        return [self._decode_actions(q, grouped[index]) for index,q in enumerate(questions)]

    def _valid(self, original, candidate, terminated=True):
        if not terminated: return False
        if not candidate or len(candidate)>max(300,4*len(original)): return False
        if len(candidate)<2 or candidate==original: return False
        if _OUTPUT_MARKER_RE.search(candidate): return False
        if _FOREIGN_INDIC_RE.search(candidate): return False
        telugu_letters = len(_TELUGU_RE.findall(candidate))
        if telugu_letters < 3 or telugu_letters / max(1, len(candidate)) < .20: return False
        if not _preserves_answer_type(original, candidate): return False
        if not _anchor_overlap(original, candidate): return False
        return entity_signature(original).issubset(entity_signature(candidate))

    def sequence_logprob(self, question, sampled, max_input=96, max_new=64):
        """Teacher-force the exact generated action (string convenience wrapper)."""
        tgt=self.tokenizer(normalize(sampled),add_special_tokens=False,return_tensors="pt",truncation=True,max_length=max(1,max_new-1))
        ids=tgt.input_ids[0].tolist()
        if self.tokenizer.eos_token_id is not None:
            ids.append(self.tokenizer.eos_token_id)
        return self.sequence_logprob_ids(question,ids,max_input)

    def sequence_logprob_ids(self, question, token_ids, max_input=96):
        if not token_ids:
            raise ValueError("A sampled reformulation must contain at least one generated token")
        src=self._source(question,max_input)
        labels=torch.tensor([token_ids],dtype=torch.long,device=self.device)
        was_training=self.model.training
        self.model.eval()
        try:
            out=self.model(**src,labels=labels)
        finally:
            self.model.train(was_training)
        logp=torch.log_softmax(out.logits,dim=-1)
        selected=logp.gather(-1,labels.unsqueeze(-1)).squeeze(-1)
        return selected.sum()

    def sequence_mean_logprob(self, question, sampled, max_input=96, max_new=64):
        """Mean token log likelihood, for length-neutral supervised warm-up."""
        tgt=self.tokenizer(normalize(sampled),add_special_tokens=False,return_tensors="pt",
                           truncation=True,max_length=max(1,max_new-1))
        ids=tgt.input_ids[0].tolist()
        if self.tokenizer.eos_token_id is not None:
            ids.append(self.tokenizer.eos_token_id)
        return self.sequence_logprob_ids(question, ids, max_input) / max(1, len(ids))

    def sequence_logprob_ids_batch(self, question, token_ids_list, max_input=96):
        """Teacher-force several sampled actions for one question in one pass.

        Returns a tensor of shape (len(token_ids_list),) with the summed
        log-probability of each sampled token sequence. Batching this step is the
        main training speedup: it replaces one full encoder-decoder forward per
        candidate with a single forward over all candidates.
        """
        if not token_ids_list:
            raise ValueError("At least one sampled reformulation is required")
        src=self._source(question,max_input)
        batch=len(token_ids_list)
        src={key:value.expand(batch,-1) for key,value in src.items()}
        max_len=max(len(ids) for ids in token_ids_list)
        labels=torch.full((batch,max_len),-100,dtype=torch.long,device=self.device)
        mask=torch.zeros((batch,max_len),dtype=torch.bool,device=self.device)
        for i,ids in enumerate(token_ids_list):
            if not ids:
                raise ValueError("A sampled reformulation must contain at least one token")
            labels[i,:len(ids)]=torch.tensor(ids,dtype=torch.long,device=self.device)
            mask[i,:len(ids)]=True
        was_training=self.model.training
        self.model.eval()
        try:
            out=self.model(**src,labels=labels)
        finally:
            self.model.train(was_training)
        logp=torch.log_softmax(out.logits,dim=-1)
        selected=logp.gather(-1,labels.clamp(min=0).unsqueeze(-1)).squeeze(-1)
        return (selected*mask).sum(dim=1)

    def sequence_logprob_batch(self, pairs, max_input=96):
        """Teacher-force several ``(question, token_ids)`` pairs in one forward pass.

        Used by the on-policy RL update: one optimizer batch contains one selected
        rewrite per question, so each row has a different source question. Sources
        are padded, target ids are padded with ``-100`` (never scored), and the
        returned tensor holds the summed log-probability of each sequence.
        """
        if not pairs:
            raise ValueError("At least one sampled reformulation is required")
        questions=[normalize(question) for question,_ in pairs]
        targets=[list(ids) for _,ids in pairs]
        if any(not ids for ids in targets):
            raise ValueError("A sampled reformulation must contain at least one generated token")
        encoded=self.tokenizer(
            [self._build_text(q) for q in questions],
            add_special_tokens=self.prompt_template is not None, return_tensors="pt",
            padding=True, truncation=True, max_length=max_input)
        src={key:value.to(self.device) for key,value in encoded.items()
             if key in {"input_ids","attention_mask"}}
        batch=len(targets)
        max_len=max(len(ids) for ids in targets)
        labels=torch.full((batch,max_len),-100,dtype=torch.long,device=self.device)
        mask=torch.zeros((batch,max_len),dtype=torch.bool,device=self.device)
        for row,ids in enumerate(targets):
            labels[row,:len(ids)]=torch.tensor(ids,dtype=torch.long,device=self.device)
            mask[row,:len(ids)]=True
        was_training=self.model.training
        self.model.eval()
        try:
            out=self.model(**src,labels=labels)
        finally:
            self.model.train(was_training)
        logp=torch.log_softmax(out.logits,dim=-1)
        selected=logp.gather(-1,labels.clamp(min=0).unsqueeze(-1)).squeeze(-1)
        return (selected*mask).sum(dim=1)

    def save_adapter(self,path):
        os.makedirs(path,exist_ok=True)
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)

In [ ]:
%%writefile /kaggle/working/activeqa/src/activeqa/qa_eval.py
"""Paired held-out QA evaluation for a reformulation policy.

The SFT script measures paraphrase quality (token-F1 against reference
rewrites). This module measures the thing that actually matters: does the
deployment pipeline answer a held-out question better after reformulation?

Every question is answered twice on its own passage:

* the original question (the do-nothing baseline), and
* the question chosen by the deployed confidence selector among the original
  plus the policy's sampled rewrites.

Selection is reference-free (span confidence + answer consistency); gold
answers are used only to compute token-F1 afterwards. ``generation_oracle_f1``
reports the best rewrite the policy produced, which separates a weak policy
from a weak selector: if the oracle rises but the selected score does not, the
selector is the bottleneck.
"""
from __future__ import annotations

import random
import re
import time

from .preprocessing import normalize
from .rewards import token_f1

TELUGU = re.compile(r"[\u0C00-\u0C7F]")
TAMIL = re.compile(r"[\u0B80-\u0BFF]")
KANNADA = re.compile(r"[\u0C80-\u0CFF]")
DEVANAGARI = re.compile(r"[\u0900-\u097F]")

CONSISTENCY_FLOOR = 0.5
CONSENSUS_FLOOR = 0.5
CONSENSUS_MIN_SUPPORT = 2
CONFIDENCE_MARGINS = (0.0, 0.1, 0.25, 0.5, 1.0)
CORRECTIVE_MARGINS = (0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 8.0)


def _script_counts(text):
    return {"te": len(TELUGU.findall(text)), "ta": len(TAMIL.findall(text)),
            "kn": len(KANNADA.findall(text)),
            "deva": len(DEVANAGARI.findall(text))}


def _bootstrap_ci(deltas, samples=1000):
    """Paired bootstrap interval for the mean per-question F1 delta."""
    if len(deltas) <= 1:
        value = round(sum(deltas) / max(1, len(deltas)), 4)
        return [value, value]
    rng = random.Random(0)
    boots = sorted(
        sum(rng.choices(deltas, k=len(deltas))) / len(deltas)
        for _ in range(samples))
    return [round(boots[int(0.025 * samples)], 4),
            round(boots[int(0.975 * samples) - 1], 4)]


def _quantile(values, q):
    """Nearest-rank quantile without adding a numpy dependency."""
    ordered = sorted(values)
    index = min(len(ordered) - 1, max(0, int(round(q * (len(ordered) - 1)))))
    return ordered[index]


def _auc(scores, labels):
    """Rank-based AUC (Mann–Whitney) for ranking positives above negatives.

    This answers a question that must come before any threshold sweep: does the
    signal order recoverable questions above the rest *at all*? An AUC of 0.5
    means no threshold on this signal can work, so a failed sweep is not a
    tuning problem. Uses the statistic directly, without a scipy dependency.
    """
    positives = [s for s, y in zip(scores, labels) if y]
    negatives = [s for s, y in zip(scores, labels) if not y]
    if not positives or not negatives:
        return 0.5
    wins = 0.0
    for positive in positives:
        for negative in negatives:
            if positive > negative:
                wins += 1.0
            elif positive == negative:
                wins += 0.5
    return round(wins / (len(positives) * len(negatives)), 4)


def deployable_rule(summary):
    """Best selection rule whose paired delta CI excludes zero, or None.

    Sweeping thresholds on one dev set inflates the maximum, so a rule is only
    deployable when its *lower* bootstrap bound is above zero. Returns
    ``(name, options)``; ``options`` maps every measured rule to
    ``(f1, ci95)`` so the report can show near misses.
    """
    original = summary["original_f1"]
    options = {
        "consistent": (summary["selected_f1"], summary.get("delta_ci95")),
        "corrective": (summary["corrective_f1"],
                       summary.get("corrective_delta_ci95")),
        "consensus": (summary["consensus_f1"],
                      summary.get("consensus_delta_ci95")),
    }
    for margin, row in summary.get("corrective_sweep", {}).items():
        options[f"corrective@{margin}"] = (row["selected_f1"],
                                           row.get("delta_ci95"))
    for key, row in summary.get("selective_sweep", {}).items():
        options[f"selective:{key}"] = (row["selected_f1"],
                                       row.get("delta_ci95"))
    passing = {
        name: value for name, value in options.items()
        if value[0] > original + 1e-9 and value[1] and value[1][0] > 0
    }
    if not passing:
        return None, options
    return max(passing, key=lambda name: passing[name][0]), options


def paired_delta_ci(records_a, records_b, key, samples=1000):
    """Bootstrap CI for mean(a[key] - b[key]) over aligned per-question rows.

    ``summary["delta_ci95"]`` compares one policy against the original
    question. That is not the interval for RL − SFT, where both policies are
    evaluated on the same rows and their per-question difference must be
    bootstrapped directly.
    """
    if len(records_a) != len(records_b):
        raise ValueError("paired CI needs the same rows for both policies")
    return _bootstrap_ci(
        [float(a[key]) - float(b[key]) for a, b in zip(records_a, records_b)],
        samples=samples)


def _margin_choice(answers, candidate_f1s, original_confidence, margin):
    """Conservative rule with an extra confidence margin over the original."""
    eligible = [0] + [i + 1 for i, cand in enumerate(candidate_f1s)
                      if cand["valid"] and cand["consistent"]
                      and cand["confidence"] >= original_confidence + margin]
    return max(eligible, key=lambda i: answers[i][2])


def _corrective_choice(answers, candidate_f1s, original_confidence, margin):
    """Highest-confidence valid rewrite, only past a confidence margin.

    The conservative rule can never change the answer, so it cannot fix a wrong
    original — the only case reformulation exists for. This rule can, but too
    small a margin makes it switch constantly and lose. Sweeping the margin
    finds out whether any amount of switching pays. Returns a 1-based index
    into ``answers`` (0 is the original).
    """
    valid = [i for i, cand in enumerate(candidate_f1s) if cand["valid"]]
    if not valid:
        return 0
    top = max(valid, key=lambda i: candidate_f1s[i]["confidence"])
    if candidate_f1s[top]["confidence"] >= original_confidence + margin:
        return top + 1
    return 0


def _consensus_choice(answers, candidate_f1s, original_answer,
                      original_confidence):
    """Pick a rewrite answer agreed on by at least two valid rewrites.

    The conservative rule can never correct a wrong original, because it
    requires the rewrite's answer to match the original's. The confidence
    override can switch, but a single confident wrong answer is enough to hurt
    (see ``corrective_f1``). Consensus is the middle ground: switch only when
    several independent rewrites land on the same different answer, and that
    answer is more confident than the original. Returns a 0-based index into
    ``candidate_f1s`` or None.
    """
    valid = [i for i, cand in enumerate(candidate_f1s) if cand["valid"]]
    if len(valid) < CONSENSUS_MIN_SUPPORT:
        return None
    clusters = []
    for index in valid:
        cand = candidate_f1s[index]
        for cluster in clusters:
            if token_f1(cand["answer"],
                        [cluster[0]["answer"]]) >= CONSENSUS_FLOOR:
                cluster.append({**cand, "index": index})
                break
        else:
            clusters.append([{**cand, "index": index}])
    best = None
    for cluster in clusters:
        if len(cluster) < CONSENSUS_MIN_SUPPORT:
            continue
        if token_f1(cluster[0]["answer"],
                    [original_answer]) >= CONSISTENCY_FLOOR:
            continue
        leader = max(cluster, key=lambda cand: cand["confidence"])
        if leader["confidence"] <= original_confidence:
            continue
        rank = (leader["confidence"], len(cluster))
        if best is None or rank > best[0]:
            best = (rank, leader["index"])
    return None if best is None else best[1]


def evaluate_qa_pairs(ref, qa, rows, *, candidates=4, limit=None,
                      max_input=96, max_new=64, do_sample=True,
                      confidence_override=0.5,
                      log=None, label="policy"):
    """Return (summary, records) for held-out QA with/without reformulation.

    ``selected_f1`` is the currently deployed rule: only rewrites whose answer
    is consistent with the original's answer may win on span confidence.
    ``corrective_f1`` additionally lets the single highest-confidence rewrite
    replace the original when it beats the original's confidence by
    ``confidence_override``. That is the case reformulation exists for (the
    original question produced a wrong span), but it is also the riskier case.
    ``consensus_f1`` switches only when at least two valid rewrites agree on a
    different answer that is more confident than the original.
    ``margin_sweep`` reports the conservative rule at several confidence
    margins over the original, so deployment does not have to guess a
    threshold. All are reported; none silently replaces the deployed rule.
    """
    rows = list(rows[:limit] if limit else rows)
    if not rows:
        raise ValueError("QA evaluation needs at least one row")
    if log:
        log(f"      QA eval [{label}]: {len(rows)} questions × "
            f"{candidates} candidates (confidence selection) …")
    started = time.perf_counter()
    records = []
    for index, row in enumerate(rows):
        question = normalize(row["question"])
        actions = ref.sample_actions(
            question, candidates, max_input, max_new, do_sample)
        texts = [question] + [action["candidate"] for action in actions]
        answers = qa.answer_batch(texts, row["context"])
        original_answer, original_confidence = answers[0][0], answers[0][2]
        original_f1 = token_f1(original_answer, row["answers"])

        candidate_f1s = []
        for action, (answer, _, confidence) in zip(actions, answers[1:]):
            candidate_f1s.append({
                "answer": answer,
                "confidence": confidence,
                "f1": token_f1(answer, row["answers"]),
                "valid": action["valid"],
                "consistent": token_f1(answer, [original_answer]) >= CONSISTENCY_FLOOR,
                "text": action["sampled_text"],
            })

        best = _margin_choice(answers, candidate_f1s, original_confidence, 0.0)
        selected_question = texts[best]
        selected_answer, selected_confidence = answers[best][0], answers[best][2]
        selected_f1 = token_f1(selected_answer, row["answers"])

        margin_f1, margin_switch = {}, {}
        for margin in CONFIDENCE_MARGINS:
            key = f"{margin:.2f}"
            index = _margin_choice(answers, candidate_f1s,
                                   original_confidence, margin)
            margin_f1[key] = token_f1(answers[index][0], row["answers"])
            margin_switch[key] = index != 0

        valid_indices = [i + 1 for i, cand in enumerate(candidate_f1s)
                         if cand["valid"]]
        top = max(valid_indices, key=lambda i: answers[i][2], default=None)
        corrective = top if (top is not None
                             and not candidate_f1s[top - 1]["consistent"]
                             and answers[top][2]
                             >= original_confidence + confidence_override) else None
        corrective_index = corrective if corrective is not None else best
        corrective_question = texts[corrective_index]
        corrective_f1 = token_f1(answers[corrective_index][0], row["answers"])

        corrective_margin_f1, corrective_margin_switch = {}, {}
        for margin in CORRECTIVE_MARGINS:
            key = f"{margin:.2f}"
            index = _corrective_choice(answers, candidate_f1s,
                                       original_confidence, margin)
            corrective_margin_f1[key] = token_f1(answers[index][0],
                                                 row["answers"])
            corrective_margin_switch[key] = index != 0

        valid_actions = [i for i, cand in enumerate(candidate_f1s)
                         if cand["valid"]]
        best_rewrite = (max(valid_actions,
                            key=lambda i: candidate_f1s[i]["confidence"])
                        if valid_actions else None)
        best_rewrite_f1 = (candidate_f1s[best_rewrite]["f1"]
                           if best_rewrite is not None else original_f1)

        consensus = _consensus_choice(answers, candidate_f1s, original_answer,
                                      original_confidence)
        consensus_index = consensus + 1 if consensus is not None else best
        consensus_question = texts[consensus_index]
        consensus_f1 = token_f1(answers[consensus_index][0], row["answers"])

        valid_f1s = [cand["f1"] for cand in candidate_f1s if cand["valid"]]
        oracle_f1 = max(valid_f1s) if valid_f1s else original_f1
        ceiling_f1 = max(original_f1, oracle_f1)
        scripts = _script_counts(selected_question)
        records.append({
            "id": row.get("id"),
            "question": question,
            "selected_question": selected_question,
            "selected_index": best,
            "switched": best != 0,
            "references": row["answers"],
            "original_answer": original_answer,
            "selected_answer": selected_answer,
            "original_f1": original_f1,
            "selected_f1": selected_f1,
            "corrective_f1": corrective_f1,
            "corrective_question": corrective_question,
            "corrective_switch": corrective_index != 0,
            "original_confidence": original_confidence,
            "selected_confidence": selected_confidence,
            "consensus_f1": consensus_f1,
            "consensus_question": consensus_question,
            "consensus_switch": consensus_index != 0,
            "best_rewrite_index": best_rewrite,
            "best_rewrite_f1": best_rewrite_f1,
            "best_rewrite_confidence": (
                candidate_f1s[best_rewrite]["confidence"]
                if best_rewrite is not None else None),
            "confidence_gain": (
                candidate_f1s[best_rewrite]["confidence"]
                - original_confidence if best_rewrite is not None else 0.0),
            "disagreement_rate": (
                sum(1 for cand in candidate_f1s
                    if cand["valid"] and not cand["consistent"])
                / max(1, sum(1 for cand in candidate_f1s if cand["valid"]))),
            "corrective_margin_f1": corrective_margin_f1,
            "corrective_margin_switch": corrective_margin_switch,
            "candidate_f1s": [cand["f1"] for cand in candidate_f1s],
            "candidate_validity": [cand["valid"] for cand in candidate_f1s],
            "candidate_texts": [cand["text"] for cand in candidate_f1s],
            "candidate_answers": [cand["answer"] for cand in candidate_f1s],
            "candidate_confidences": [cand["confidence"] for cand in candidate_f1s],
            "margin_f1": margin_f1,
            "margin_switch": margin_switch,
            "generation_oracle_f1": oracle_f1,
            "ceiling_f1": ceiling_f1,
            "valid": True if best == 0 else candidate_f1s[best - 1]["valid"],
            "scripts": scripts,
        })
        if log and (index + 1) % 25 == 0:
            done = records
            log(f"      QA eval [{label}] {index + 1}/{len(rows)}  "
                f"orig={sum(r['original_f1'] for r in done) / len(done):.4f}  "
                f"selected={sum(r['selected_f1'] for r in done) / len(done):.4f}")

    n = max(1, len(records))
    candidate_count = sum(len(r["candidate_f1s"]) for r in records)
    valid_count = sum(sum(r["candidate_validity"]) for r in records)
    switched = [r for r in records if r["switched"]]
    summary = {
        "label": label,
        "examples": len(records),
        "candidates_per_question": candidates,
        "original_f1": round(sum(r["original_f1"] for r in records) / n, 4),
        "selected_f1": round(sum(r["selected_f1"] for r in records) / n, 4),
        "corrective_f1": round(sum(r["corrective_f1"] for r in records) / n, 4),
        "consensus_f1": round(sum(r["consensus_f1"] for r in records) / n, 4),
        "generation_oracle_f1": round(
            sum(r["generation_oracle_f1"] for r in records) / n, 4),
        "improved_rate": round(
            sum(r["selected_f1"] > r["original_f1"] + 1e-9 for r in records) / n, 4),
        "harmed_rate": round(
            sum(r["selected_f1"] < r["original_f1"] - 1e-9 for r in records) / n, 4),
        "switch_rate": round(len(switched) / n, 4),
        "corrective_switch_rate": round(
            sum(r["corrective_switch"] for r in records) / n, 4),
        "corrective_harmed_rate": round(
            sum(r["corrective_f1"] < r["original_f1"] - 1e-9 for r in records) / n, 4),
        "consensus_switch_rate": round(
            sum(r["consensus_switch"] for r in records) / n, 4),
        "consensus_harmed_rate": round(
            sum(r["consensus_f1"] < r["original_f1"] - 1e-9 for r in records) / n, 4),
        "consensus_improved_rate": round(
            sum(r["consensus_f1"] > r["original_f1"] + 1e-9 for r in records) / n, 4),
        "confidence_override": confidence_override,
        "mean_confidence_delta": round(
            sum(r["selected_confidence"] - r["original_confidence"]
                for r in records) / n, 4),
        "valid_generation_rate": round(valid_count / max(1, candidate_count), 4),
        "distinct_selected_ratio": round(
            len({r["selected_question"] for r in records}) / n, 4),
        "ends_with_question_mark": round(
            sum(r["selected_question"].strip().endswith("?") for r in records) / n, 4),
        "tamil_spill_outputs": sum(1 for r in records if r["scripts"]["ta"]),
        "kannada_spill_outputs": sum(1 for r in records if r["scripts"]["kn"]),
        "devanagari_spill_outputs": sum(1 for r in records if r["scripts"]["deva"]),
        "wall_seconds": round(time.perf_counter() - started, 1),
    }
    summary["delta_f1"] = round(summary["selected_f1"] - summary["original_f1"], 4)
    summary["corrective_delta_f1"] = round(
        summary["corrective_f1"] - summary["original_f1"], 4)
    summary["consensus_delta_f1"] = round(
        summary["consensus_f1"] - summary["original_f1"], 4)
    summary["delta_ci95"] = _bootstrap_ci(
        [record["selected_f1"] - record["original_f1"] for record in records])
    summary["corrective_delta_ci95"] = _bootstrap_ci(
        [record["corrective_f1"] - record["original_f1"] for record in records])
    summary["consensus_delta_ci95"] = _bootstrap_ci(
        [record["consensus_f1"] - record["original_f1"] for record in records])
    summary["generation_oracle_delta"] = round(
        summary["generation_oracle_f1"] - summary["original_f1"], 4)

    margin_sweep = {}
    for margin in CONFIDENCE_MARGINS:
        key = f"{margin:.2f}"
        values = [record["margin_f1"][key] for record in records]
        margin_sweep[key] = {
            "selected_f1": round(sum(values) / n, 4),
            "delta_f1": round(sum(values) / n - summary["original_f1"], 4),
            "switch_rate": round(
                sum(record["margin_switch"][key] for record in records) / n, 4),
            "improved_rate": round(
                sum(value > record["original_f1"] + 1e-9
                    for value, record in zip(values, records)) / n, 4),
            "harmed_rate": round(
                sum(value < record["original_f1"] - 1e-9
                    for value, record in zip(values, records)) / n, 4),
        }
    summary["margin_sweep"] = margin_sweep
    summary["best_margin"] = max(
        margin_sweep, key=lambda key: margin_sweep[key]["selected_f1"])

    corrective_sweep = {}
    for margin in CORRECTIVE_MARGINS:
        key = f"{margin:.2f}"
        values = [record["corrective_margin_f1"][key] for record in records]
        corrective_sweep[key] = {
            "selected_f1": round(sum(values) / n, 4),
            "delta_f1": round(sum(values) / n - summary["original_f1"], 4),
            "switch_rate": round(
                sum(record["corrective_margin_switch"][key]
                    for record in records) / n, 4),
            "improved_rate": round(
                sum(value > record["original_f1"] + 1e-9
                    for value, record in zip(values, records)) / n, 4),
            "harmed_rate": round(
                sum(value < record["original_f1"] - 1e-9
                    for value, record in zip(values, records)) / n, 4),
            "delta_ci95": _bootstrap_ci(
                [record["corrective_margin_f1"][key] - record["original_f1"]
                 for record in records]),
        }
    summary["corrective_sweep"] = corrective_sweep
    summary["best_corrective_margin"] = max(
        corrective_sweep,
        key=lambda key: corrective_sweep[key]["selected_f1"])

    confidences_for_gate = [record["original_confidence"] for record in records]
    selective_sweep = {}
    for quantile in (0.25, 0.5, 0.75):
        threshold = _quantile(confidences_for_gate, quantile)
        for margin in CORRECTIVE_MARGINS:
            margin_key = f"{margin:.2f}"
            values, switches = [], 0
            for record in records:
                if (record["original_confidence"] <= threshold
                        and record["corrective_margin_switch"][margin_key]):
                    values.append(record["corrective_margin_f1"][margin_key])
                    switches += 1
                else:
                    values.append(record["original_f1"])
            selective_sweep[f"q{quantile:.2f}/m{margin_key}"] = {
                "selected_f1": round(sum(values) / n, 4),
                "delta_f1": round(sum(values) / n - summary["original_f1"], 4),
                "switch_rate": round(switches / n, 4),
                "delta_ci95": _bootstrap_ci(
                    [value - record["original_f1"]
                     for value, record in zip(values, records)]),
            }
    summary["selective_sweep"] = selective_sweep
    summary["best_selective"] = max(
        selective_sweep,
        key=lambda key: selective_sweep[key]["selected_f1"])

    def _mean(values):
        return sum(values) / len(values) if values else 0.0

    recoverable = [record for record in records
                   if record["ceiling_f1"] > record["original_f1"] + 1e-9]
    not_recoverable = [record for record in records
                       if record["ceiling_f1"] <= record["original_f1"] + 1e-9]
    summary["ceiling_f1"] = round(
        sum(record["ceiling_f1"] for record in records) / n, 4)
    summary["ceiling_delta_f1"] = round(
        summary["ceiling_f1"] - summary["original_f1"], 4)
    summary["recoverable_rate"] = round(len(recoverable) / n, 4)
    summary["recoverable_mean_gain"] = round(_mean(
        [record["ceiling_f1"] - record["original_f1"]
         for record in recoverable]), 4)
    summary["recoverable_original_confidence"] = round(_mean(
        [record["original_confidence"] for record in recoverable]), 4)
    summary["nonrecoverable_original_confidence"] = round(_mean(
        [record["original_confidence"] for record in not_recoverable]), 4)

    labels = [record["ceiling_f1"] > record["original_f1"] + 1e-9
              for record in records]
    summary["signal_auc"] = {
        "low_original_confidence": _auc(
            [-record["original_confidence"] for record in records], labels),
        "confidence_gain": _auc(
            [record["confidence_gain"] for record in records], labels),
        "rewrite_disagreement": _auc(
            [record["disagreement_rate"] for record in records], labels),
    }

    confidences = [record["original_confidence"] for record in records]
    gate_sweep = {}
    for quantile in (0.0, 0.1, 0.25, 0.5, 0.75, 1.0):
        threshold = _quantile(confidences, quantile)
        gated = [record for record in records
                 if record["original_confidence"] <= threshold]
        values = [(record["best_rewrite_f1"]
                   if record["original_confidence"] <= threshold
                   else record["original_f1"]) for record in records]
        gated_recoverable = [record for record in gated
                             if record["ceiling_f1"]
                             > record["original_f1"] + 1e-9]
        gate_sweep[f"{quantile:.2f}"] = {
            "threshold": round(threshold, 4),
            "examples": len(gated),
            "gate_rate": round(len(gated) / n, 4),
            "selected_f1": round(sum(values) / n, 4),
            "delta_f1": round(sum(values) / n - summary["original_f1"], 4),
            "gated_original_f1": round(_mean(
                [record["original_f1"] for record in gated]), 4),
            "gated_selected_f1": round(_mean(
                [record["best_rewrite_f1"] for record in gated]), 4),
            "gated_ceiling_f1": round(_mean(
                [record["ceiling_f1"] for record in gated]), 4),
            "gated_recoverable_rate": round(
                len(gated_recoverable) / max(1, len(gated)), 4),
            "switch_rate": round(sum(
                record["best_rewrite_index"] is not None
                for record in gated) / max(1, len(gated)), 4),
            "harmed_rate": round(sum(
                value < record["original_f1"] - 1e-9
                for value, record in zip(values, records)) / n, 4),
        }
    summary["gate_sweep"] = gate_sweep
    summary["best_gate"] = max(
        gate_sweep, key=lambda key: gate_sweep[key]["selected_f1"])
    summary["switched_mean_lift"] = round(
        sum(r["selected_f1"] - r["original_f1"] for r in switched)
        / max(1, len(switched)), 4)
    return summary, records

In [ ]:
%%writefile /kaggle/working/activeqa/scripts/kaggle_run_rl.py
"""Telugu reformulator RL stage: starting policy → QA-reward RFT → paired held-out QA.

This is stage 2 of the one-command pipeline (stage 1 is scripts/colab_run_sft.py).
It uses the frozen Telugu QA model as the reward, runs on-policy
rejection-sampling fine-tuning, then evaluates the starting and RL policies on
the same held-out questions:

    python scripts/kaggle_run_rl.py                 # RL on top of the SFT adapter
    python scripts/kaggle_run_rl.py --from-base     # RL from the raw checkpoint
    python scripts/kaggle_run_rl.py --from-base --model bigscience/mt0-base

The starting policy is the SFT LoRA adapter when available, or the raw
checkpoint with a fresh LoRA when ``--from-base`` is passed.

Checkpoint selection guarantees the run can never ship a policy worse than the
one it started with, no matter how noisy RL training is: every
``--checkpoint-every`` optimizer updates (default 12) the current adapter is
saved to <out>/probe/ and measured with ``evaluate_qa_pairs`` on a dev subset
(``--checkpoint-eval-questions``, default 80). The trained end-state is probed
too. The probe with the best subset ``selected_f1`` — including the starting
policy itself — is reloaded and shipped.

The number that matters is `selected_f1` in <out>/qa_eval.json: the answer
token-F1 of the deployed confidence-selected reformulation, measured against
the original question on the same passage.

Everything is written to the run directory (models/indicbartss-reformulator-lora-rl):
  adapter/            RL LoRA adapter (shipped policy; the starting policy when
                      no checkpoint beat it)
  run.log             full transcript
  run_status.json     last completed stage (survives a crash)
  rl_metrics.json     reward/selection/validity statistics, checkpoint history
  rl_samples.json     per-question training selections (raw, unfiltered)
  qa_eval.json        start vs RL held-out QA summary + accept/reject verdict
  qa_eval_samples.json per-question predictions for both policies
"""
from __future__ import annotations

import argparse
import json
import os
import random
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

MODEL_ID = "ai4bharat/MultiIndicParaphraseGenerationSS"
PINNED_TRANSFORMERS = "4.57.6"
LOCAL_MODEL_DIR = ROOT / "models" / "MultiIndicParaphraseGenerationSS"
QA_MODEL = "l3cube-pune/telugu-question-answering-squad-bert"
SFT_ADAPTER = ROOT / "models" / "indicbartss-reformulator-lora" / "adapter"
RL_DIR = ROOT / "models" / "indicbartss-reformulator-lora-rl"

LOG_HANDLE: dict[str, Path | None] = {"path": None}


def log(message: str) -> None:
    print(message, flush=True)
    path = LOG_HANDLE["path"]
    if path:
        with open(path, "a", encoding="utf-8") as stream:
            stream.write(str(message) + "\n")


def record_status(path: Path, stage: str, **fields) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {}
    if path.exists():
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            payload = {}
    payload.update({
        "stage": stage,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    })
    payload.update(fields)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False),
                    encoding="utf-8")


def parse_args() -> argparse.Namespace:
    ap = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--sft-adapter", default=str(SFT_ADAPTER),
                    help="LoRA adapter produced by scripts/colab_run_sft.py; "
                         "ignored with --from-base")
    ap.add_argument("--from-base", action="store_true",
                    help="train RL directly from the raw checkpoint with a "
                         "fresh LoRA — no SFT stage required")
    ap.add_argument("--model", default=MODEL_ID,
                    help="any HF seq2seq checkpoint for from-base RL "
                         "(e.g. bigscience/mt0-base; only used with --from-base "
                         "unless you are sure the adapter matches)")
    ap.add_argument("--rl-lora-rank", type=int, default=32,
                    help="rank of the fresh LoRA when --from-base")
    ap.add_argument("--rl-lora-alpha", type=int, default=64,
                    help="alpha of the fresh LoRA when --from-base")
    ap.add_argument("--rl-mlp", action="store_true",
                    help="also attach LoRA to the feed-forward layers")
    ap.add_argument("--rl-prompt", default=None,
                    help="override the paraphrase instruction prompt used by "
                         "prompted (--model) checkpoints; {question} is the "
                         "placeholder for the input question")
    ap.add_argument("--preserve-weight", type=float, default=0.2,
                    help="reward weight for entity preservation (0.1 = old "
                         "default); higher punishes entity-text drift")
    ap.add_argument("--checkpoint-eval-questions", type=int, default=200,
                    help="dev questions evaluated at every RL checkpoint "
                         "(checkpoint selection; 0 disables)")
    ap.add_argument("--checkpoint-every", type=int, default=8,
                    help="run a checkpoint eval every N optimizer updates")
    ap.add_argument("--out", default=str(RL_DIR),
                    help="run directory; the RL adapter is written to <out>/adapter")
    ap.add_argument("--train", default="data/indic_rag_te/train.jsonl")
    ap.add_argument("--dev", default="data/indic_rag_te/dev.jsonl")
    ap.add_argument("--questions", type=int, default=0,
                    help="training questions per epoch (0 = all; train.jsonl "
                         "has 804)")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--candidates", type=int, default=6)
    ap.add_argument("--batch-questions", type=int, default=8,
                    help="selected targets per optimizer update")
    ap.add_argument("--generate-batch", type=int, default=8,
                    help="questions generated together (speed); lower if OOM")
    ap.add_argument("--lr", type=float, default=7e-5)
    ap.add_argument("--margin", type=float, default=0.0,
                    help="required reward gain over the original question")
    ap.add_argument("--conf-slack", type=float, default=0.0,
                    help="allowed confidence deficit vs the original question "
                         "(0.0 matches the deployed selector; positive values "
                         "train on rewrites deployment would not choose)")
    ap.add_argument("--max-input", type=int, default=96)
    ap.add_argument("--max-new", type=int, default=64)
    ap.add_argument("--eval-questions", type=int, default=200)
    ap.add_argument("--eval-candidates", type=int, default=4)
    ap.add_argument("--conf-override", type=float, default=0.5,
                    help="confidence margin that lets a corrective rewrite replace "
                         "the original's answer (reported as corrective_f1)")
    ap.add_argument("--no-eval", action="store_true",
                    help="skip both held-out QA evaluations (fast training only)")
    ap.add_argument("--eval-only", action="store_true",
                    help="evaluate the SFT adapter and the selector sweep on the "
                         "held-out set, then stop (no RL training)")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--skip-install", action="store_true")
    ap.add_argument("--smoke", action="store_true",
                    help="CPU-friendly code check: 2 questions, 2 candidates")
    return ap.parse_args()


def installed_transformers() -> str:
    """Version on disk. Must be checked before importing transformers."""
    try:
        from importlib.metadata import version
        return version("transformers")
    except Exception:
        return ""


def ensure_deps() -> None:
    """Install the exact pin, then restart before transformers is imported.

    Kaggle now preinstalls transformers 5.x. A ">= 4.57" check accepts it, but
    5.x loads this checkpoint without tying its shared embedding table, which
    collapses every output to the same string. Only an exact version match is
    safe, and the restart avoids keeping already-imported 5.x modules alive.
    """
    current = installed_transformers()
    if current == PINNED_TRANSFORMERS:
        log(f"[deps] transformers {current} already installed")
        return
    log(f"[deps] installing transformers {PINNED_TRANSFORMERS} "
        f"(found {current or 'none'}) …")
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        f"transformers=={PINNED_TRANSFORMERS}", "peft>=0.13,<0.22",
        "sentencepiece>=0.2.0", "protobuf>=4.25", "tqdm>=4.66",
        "huggingface_hub>=0.23,<1.0",
    ], stdout=subprocess.DEVNULL)
    log("[deps] done; restarting so the pinned build is the one imported")
    os.execv(sys.executable,
             [sys.executable, str(Path(__file__).resolve())] + sys.argv[1:])


def resolve_model_path(model_id: str = MODEL_ID) -> str:
    """Materialize ``model_id`` on disk and return the local snapshot path.

    Only the PyTorch weights are downloaded — many HF repos carry ONNX/TF/
    rust duplicates that are several GB each and useless to this pipeline.
    The download location is the candidate directory with the most free
    space (on Kaggle ``/kaggle/tmp`` is a tens-of-GB scratch disk while
    ``/kaggle/working`` can be nearly full).
    """
    import shutil
    candidates = [ROOT / "models", Path("/kaggle/tmp")]
    best, best_free = candidates[0], -1
    for cand in candidates:
        try:
            free = shutil.disk_usage(str(cand)).free
        except OSError:
            free = -1
        if free > best_free:
            best, best_free = cand, free
    local = best / model_id.split("/")[-1]
    if local.is_dir() and (list(local.glob("*.safetensors"))
                           or list(local.glob("*.bin"))):
        return str(local)
    from huggingface_hub import HfApi, snapshot_download
    ignore = ["*.msgpack", "flax_model*", "tf_model*",
              "rust_model*", "*.onnx", "onnx/*", "*.mlmodel",
              "*.h5", "*.png", "*.jpg", "*.pdf"]
    try:
        files = HfApi().list_repo_files(model_id)
        if any(f.endswith(".safetensors") for f in files):
            ignore.insert(0, "*.bin")
    except Exception:
        pass
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    return snapshot_download(repo_id=model_id, local_dir=str(local),
                             ignore_patterns=ignore)


def adapter_ready(path: Path) -> bool:
    return (path / "adapter_config.json").is_file() and (
        (path / "adapter_model.safetensors").is_file()
        or (path / "adapter_model.bin").is_file())


def load_rows(path: str, limit: int, seed: int) -> list[dict]:
    from activeqa.data import _load_jsonl_qa
    rows = _load_jsonl_qa(path)
    random.Random(seed).shuffle(rows)
    if limit and limit > 0:
        rows = rows[:limit]
    return rows


def paired_table(start: dict, rl: dict, start_name: str = "SFT") -> str:
    keys = ("original_f1", "selected_f1", "corrective_f1", "consensus_f1",
            "generation_oracle_f1", "improved_rate", "harmed_rate",
            "switch_rate", "corrective_switch_rate", "corrective_harmed_rate",
            "consensus_switch_rate", "consensus_harmed_rate",
            "valid_generation_rate", "distinct_selected_ratio")
    lines = [f"  {'metric':28s} {start_name:>9s} {'RL':>9s} {'delta':>9s}"]
    for key in keys:
        delta = rl[key] - start[key]
        lines.append(f"  {key:28s} {start[key]:>9.4f} {rl[key]:>9.4f} {delta:>+9.4f}")
    return "\n".join(lines)


def main() -> None:
    args = parse_args()
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    LOG_HANDLE["path"] = out_dir / "run.log"
    status_path = out_dir / "run_status.json"
    record_status(status_path, "started", args=vars(args))

    if args.smoke:
        args.skip_install = installed_transformers() == PINNED_TRANSFORMERS
        args.questions = 2
        args.epochs = 1
        args.candidates = 2
        args.batch_questions = 1
        args.generate_batch = 1
        args.eval_questions = 2
        args.eval_candidates = 2
        args.checkpoint_eval_questions = 2
        args.checkpoint_every = 1
        args.max_new = 16

    sft_adapter = Path(args.sft_adapter)
    if args.eval_only and args.from_base:
        raise SystemExit("--eval-only evaluates the SFT adapter; it is "
                         "incompatible with --from-base")
    if not args.from_base and args.model != MODEL_ID:
        raise SystemExit("--model applies to --from-base; the SFT adapter was "
                         f"trained on {MODEL_ID}")
    if not args.from_base and not adapter_ready(sft_adapter):
        raise SystemExit(
            f"SFT adapter not found at {sft_adapter}. Run scripts/colab_run_sft.py "
            "first, use scripts/kaggle_run_all.py for both stages, or pass "
            "--from-base to train RL without the SFT adapter.")

    import torch
    if not args.skip_install:
        ensure_deps()
    import transformers
    import peft
    if (transformers.__version__ != PINNED_TRANSFORMERS
            and not args.smoke):
        raise SystemExit(
            f"transformers {transformers.__version__} is loaded but this "
            f"pipeline requires {PINNED_TRANSFORMERS}. Rerun without "
            "--skip-install (or reinstall the pin) in a fresh process.")
    log(f"[1/6] RL stage — transformers {transformers.__version__}  "
        f"peft {peft.__version__}  torch {torch.__version__}")

    use_cuda = torch.cuda.is_available()
    device = "cuda" if use_cuda else "cpu"
    if not use_cuda and not args.smoke:
        raise SystemExit("No CUDA GPU found. Select a GPU runtime (T4) on Kaggle/Colab.")
    if use_cuda:
        log(f"      GPU: {torch.cuda.get_device_name(0)}  "
            f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
    random.seed(args.seed)
    torch.manual_seed(args.seed)

    sys.path.insert(0, str(ROOT / "scripts"))
    from activeqa.qa_model import TeluguQA
    from activeqa.qa_eval import evaluate_qa_pairs, paired_delta_ci, deployable_rule
    from activeqa.reformulator import Reformulator
    from activeqa.rft import rft_on_policy

    model_path = resolve_model_path(args.model)
    log(f"[2/6] Reformulator {args.model}")
    started = time.perf_counter()
    if args.from_base:
        log(f"      --from-base: no SFT adapter; fresh LoRA "
            f"(r={args.rl_lora_rank}, alpha={args.rl_lora_alpha}"
            f"{', +mlp' if args.rl_mlp else ''})")
        ref = Reformulator(model_path, cache_dir="models", device=device,
                           rank=args.rl_lora_rank, alpha=args.rl_lora_alpha,
                           include_mlp=args.rl_mlp, prompt=args.rl_prompt)
    else:
        log(f"      loading SFT adapter {sft_adapter}")
        ref = Reformulator(model_path, cache_dir="models", device=device,
                           adapter_path=str(sft_adapter))
    log(f"      trainable params: {ref.trainable_count:,}  "
        f"({time.perf_counter() - started:.1f}s)")
    record_status(status_path, "policy_loaded",
                  adapter=str(sft_adapter) if not args.from_base else None,
                  from_base=args.from_base, model=args.model,
                  trainable=ref.trainable_count)

    qa_started = time.perf_counter()
    log(f"[3/6] Frozen QA reward model {QA_MODEL}")
    qa = TeluguQA(QA_MODEL, cache_dir="models", device=device)
    log(f"      loaded in {time.perf_counter() - qa_started:.1f}s")

    if args.no_eval:
        sft_eval = rl_eval = None
        sft_records = rl_records = []
    else:
        start_name = "base" if args.from_base else "sft"
        dev_rows = load_rows(args.dev, args.eval_questions, args.seed)
        log(f"[4/6] Held-out QA eval — starting policy "
            f"({'raw checkpoint' if args.from_base else 'SFT adapter'}), "
            f"{len(dev_rows)} questions")
        torch.manual_seed(args.seed)
        sft_eval, sft_records = evaluate_qa_pairs(
            ref, qa, dev_rows, candidates=args.eval_candidates,
            limit=args.eval_questions, max_input=args.max_input,
            max_new=args.max_new, confidence_override=args.conf_override,
            log=log, label=start_name)
        record_status(status_path, "pre_eval_done",
                      selected_f1=sft_eval["selected_f1"])
        log(f"      {start_name} selected_f1={sft_eval['selected_f1']:.4f} "
            f"(original {sft_eval['original_f1']:.4f}, "
            f"generation oracle {sft_eval['generation_oracle_f1']:.4f})")
        if args.eval_only:
            print("\n── SFT selector diagnostics (no training) ─────────────────────")
            print(f"  original question        {sft_eval['original_f1']:.4f}")
            print(f"  consistent (deployed)    {sft_eval['selected_f1']:.4f}  "
                  f"switch={sft_eval['switch_rate']:.3f}  "
                  f"harmed={sft_eval['harmed_rate']:.3f}")
            print(f"  confident correction     {sft_eval['corrective_f1']:.4f}  "
                  f"switch={sft_eval['corrective_switch_rate']:.3f}  "
                  f"harmed={sft_eval['corrective_harmed_rate']:.3f}")
            print(f"  answer consensus         {sft_eval['consensus_f1']:.4f}  "
                  f"switch={sft_eval['consensus_switch_rate']:.3f}  "
                  f"harmed={sft_eval['consensus_harmed_rate']:.3f}")
            print("  confidence-margin sweep (consistent rule):")
            for margin, row in sft_eval["margin_sweep"].items():
                print(f"    +{margin}  f1={row['selected_f1']:.4f}  "
                      f"switch={row['switch_rate']:.3f}  "
                      f"harmed={row['harmed_rate']:.3f}")
            print(f"  best margin on this dev set: {sft_eval['best_margin']} "
                  f"(diagnostic only; confirm on a second split)")
            print(f"  ceiling (best of original+rewrites) "
                  f"{sft_eval['ceiling_f1']:.4f} "
                  f"({sft_eval['ceiling_delta_f1']:+.4f}); "
                  f"recoverable {sft_eval['recoverable_rate']:.3f} "
                  f"(gain {sft_eval['recoverable_mean_gain']:+.4f}, "
                  f"orig conf {sft_eval['recoverable_original_confidence']:+.2f} "
                  f"vs {sft_eval['nonrecoverable_original_confidence']:+.2f})")
            auc = sft_eval["signal_auc"]
            print(f"  signal AUC for recoverability: "
                  f"low_original_conf {auc['low_original_confidence']:.3f}  "
                  f"confidence_gain {auc['confidence_gain']:.3f}  "
                  f"disagreement {auc['rewrite_disagreement']:.3f}")
            print("  corrective sweep (highest-confidence rewrite past a "
                  "confidence margin):")
            for margin, row in sft_eval["corrective_sweep"].items():
                print(f"    +{margin}  f1={row['selected_f1']:.4f}  "
                      f"delta={row['delta_f1']:+.4f}  "
                      f"CI[{row['delta_ci95'][0]:+.4f},{row['delta_ci95'][1]:+.4f}]  "
                      f"switch={row['switch_rate']:.3f}  "
                      f"imp={row['improved_rate']:.3f}  "
                      f"harm={row['harmed_rate']:.3f}")
            print(f"  best corrective margin on this dev set: "
                  f"{sft_eval['best_corrective_margin']} "
                  f"({sft_eval['corrective_sweep'][sft_eval['best_corrective_margin']]['selected_f1']:.4f})")
            print("  low-confidence gate sweep (best valid rewrite when the "
                  "original answer is uncertain):")
            for quantile, row in sft_eval["gate_sweep"].items():
                print(f"    q{quantile} ≤{row['threshold']:+.3f}  "
                      f"f1={row['selected_f1']:.4f}  "
                      f"gated={row['examples']:3d}  "
                      f"orig {row['gated_original_f1']:.4f}  "
                      f"ceil {row['gated_ceiling_f1']:.4f}  "
                      f"recov={row['gated_recoverable_rate']:.3f}")
            best = sft_eval["selective_sweep"][sft_eval["best_selective"]]
            print(f"  best gate+margin combo: {sft_eval['best_selective']}  "
                  f"f1={best['selected_f1']:.4f} ({best['delta_f1']:+.4f})  "
                  f"CI[{best['delta_ci95'][0]:+.4f},{best['delta_ci95'][1]:+.4f}]  "
                  f"switch={best['switch_rate']:.3f}")
            rule, rule_options = deployable_rule(sft_eval)
            if rule:
                print(f"  DEPLOYABLE RULE: {rule}  f1={rule_options[rule][0]:.4f}  "
                      f"CI[{rule_options[rule][1][0]:+.4f},"
                      f"{rule_options[rule][1][1]:+.4f}]  "
                      f"(original {sft_eval['original_f1']:.4f})")
            else:
                print("  DEPLOYABLE RULE: none — every selector's paired 95% CI "
                      "includes zero; do not rewrite.")
            (out_dir / "qa_eval_sft_only.json").write_text(
                json.dumps({"sft": sft_eval}, indent=2, ensure_ascii=False),
                encoding="utf-8")
            (out_dir / "qa_eval_samples_sft_only.json").write_text(
                json.dumps({"sft": sft_records}, indent=2,
                           ensure_ascii=False), encoding="utf-8")
            record_status(status_path, "eval_only_done",
                          selected_f1=sft_eval["selected_f1"],
                          consensus_f1=sft_eval["consensus_f1"])
            return

    import shutil

    cp_rows = []
    starting_subset_f1 = None
    checkpoint_history: list[dict] = []
    best_checkpoint: dict = {"updates": 0, "subset_selected_f1": None,
                             "path": None}
    if not args.no_eval and args.checkpoint_eval_questions > 0:
        cp_rows = load_rows(args.dev, args.checkpoint_eval_questions, args.seed)
    if cp_rows:
        log(f"      checkpoint selection on {len(cp_rows)} dev questions, "
            f"every {args.checkpoint_every} updates")
        torch.manual_seed(args.seed)
        ref.model.eval()
        subset_base, _ = evaluate_qa_pairs(
            ref, qa, cp_rows, candidates=args.eval_candidates,
            limit=len(cp_rows), max_input=args.max_input,
            max_new=args.max_new, confidence_override=args.conf_override,
            log=log, label="checkpoint-baseline")
        ref.model.train()
        starting_subset_f1 = subset_base["selected_f1"]
        best_checkpoint["subset_selected_f1"] = starting_subset_f1
        checkpoint_history.append(
            {"updates": 0, "state": "starting policy",
             "selected_f1": starting_subset_f1})
        log(f"      starting policy subset selected_f1: {starting_subset_f1:.4f}")

    def checkpoint_hook(n_updates: int) -> None:
        if not cp_rows or n_updates % max(1, args.checkpoint_every):
            return
        probe = out_dir / "probe" / f"ckpt_{n_updates:04d}"
        ref.save_adapter(str(probe))
        ref.model.eval()
        torch.manual_seed(args.seed)
        summary, _ = evaluate_qa_pairs(
            ref, qa, cp_rows, candidates=args.eval_candidates,
            limit=len(cp_rows), max_input=args.max_input,
            max_new=args.max_new, confidence_override=args.conf_override,
            log=log, label=f"checkpoint@{n_updates}")
        ref.model.train()
        f1 = summary["selected_f1"]
        checkpoint_history.append({"updates": n_updates, "state": str(probe),
                                   "selected_f1": f1})
        if f1 > best_checkpoint["subset_selected_f1"]:
            best_checkpoint.update(updates=n_updates, subset_selected_f1=f1,
                                   path=str(probe))
            log(f"      checkpoint {n_updates} is the new best subset "
                f"selected_f1: {f1:.4f}")

    train_rows = load_rows(args.train, args.questions, args.seed)
    log(f"[5/6] RL (RFT) on {len(train_rows)} questions × {args.epochs} epoch(s)"
        + ("  [--from-base]" if args.from_base else ""))
    metrics, train_records = rft_on_policy(
        ref, qa, train_rows,
        candidates=args.candidates,
        batch_questions=args.batch_questions,
        epochs=args.epochs,
        learning_rate=args.lr,
        margin=args.margin,
        confidence_slack=args.conf_slack,
        max_input=args.max_input,
        max_new=args.max_new,
        generate_batch=args.generate_batch,
        seed=args.seed,
        after_update=checkpoint_hook if cp_rows else None,
        skip_solved=True,
        preserve_weight=args.preserve_weight,
        log=log)
    record_status(status_path, "train_done",
                  optimizer_steps=metrics["optimizer_steps"],
                  selected_rate=metrics["selected_rate"])

    if metrics["optimizer_steps"] == 0:
        log("      WARNING: no qualifying rewrite was found; the starting policy "
            "was not changed. Lower --margin / --conf-slack or raise --candidates.")
        record_status(status_path, "no_updates")
        (out_dir / "rl_metrics.json").write_text(
            json.dumps({"training": metrics,
                        "sft_adapter": str(sft_adapter),
                        "from_base": args.from_base},
                       indent=2, ensure_ascii=False), encoding="utf-8")
        print("\nRL produced no updates — keeping the starting policy unchanged.")
        return

    adapter_dir = out_dir / "adapter"
    used_checkpoint_selection = bool(cp_rows and not args.no_eval)
    reverted = False
    if used_checkpoint_selection:
        final_probe = out_dir / "probe" / "ckpt_final"
        ref.save_adapter(str(final_probe))
        ref.model.eval()
        torch.manual_seed(args.seed)
        summary, _ = evaluate_qa_pairs(
            ref, qa, cp_rows, candidates=args.eval_candidates,
            limit=len(cp_rows), max_input=args.max_input,
            max_new=args.max_new, confidence_override=args.conf_override,
            log=log, label="checkpoint@final")
        f1 = summary["selected_f1"]
        checkpoint_history.append({
            "updates": metrics["optimizer_steps"], "state": str(final_probe),
            "selected_f1": f1})
        if f1 > best_checkpoint["subset_selected_f1"]:
            best_checkpoint.update(updates=metrics["optimizer_steps"],
                                   subset_selected_f1=f1,
                                   path=str(final_probe))
        if best_checkpoint["path"] is None:
            reverted = True
            log(f"      RL never beat the starting policy on the checkpoint "
                f"subset ({starting_subset_f1:.4f}); shipping the starting "
                "policy unchanged")
        ref_device = str(ref.device)
        del ref
        torch.cuda.empty_cache()
        if reverted:
            if args.from_base:
                ref = Reformulator(model_path, cache_dir="models",
                                   device=ref_device,
                                   rank=args.rl_lora_rank,
                                   alpha=args.rl_lora_alpha,
                                   include_mlp=args.rl_mlp,
                                   prompt=args.rl_prompt)
            else:
                ref = Reformulator(model_path, cache_dir="models",
                                   device=ref_device,
                                   adapter_path=str(sft_adapter))
            ref.model.eval()
        else:
            ref = Reformulator(model_path, cache_dir="models",
                               device=ref_device,
                               adapter_path=str(best_checkpoint["path"]))
            ref.model.eval()
            log(f"      shipping checkpoint at {best_checkpoint['updates']} "
                f"updates (subset selected_f1 "
                f"{best_checkpoint['subset_selected_f1']:.4f})")
        shutil.rmtree(out_dir / "probe", ignore_errors=True)
    if args.from_base and reverted:
        rl_adapter_path = None      # the starting (adapter-free) policy ships
    else:
        ref.save_adapter(str(adapter_dir))
        rl_adapter_path = str(adapter_dir)
        log(f"[6/6] RL adapter saved: {adapter_dir}")
    record_status(status_path, "adapter_saved",
                  optimizer_steps=metrics["optimizer_steps"],
                  reverted_to_start=reverted)
    (out_dir / "rl_metrics.json").write_text(json.dumps({
        "model": args.model,
        "training": metrics,
        "sft_adapter": str(sft_adapter),
        "from_base": args.from_base,
        "rl_adapter": rl_adapter_path,
        "checkpoint_selection": {
            "enabled": used_checkpoint_selection,
            "subset_questions": len(cp_rows),
            "checkpoint_every": args.checkpoint_every,
            "starting_subset_selected_f1": starting_subset_f1,
            "best": best_checkpoint,
            "history": checkpoint_history,
            "reverted_to_start": reverted,
        },
    }, indent=2, ensure_ascii=False), encoding="utf-8")
    (out_dir / "rl_samples.json").write_text(
        json.dumps(train_records, indent=2, ensure_ascii=False), encoding="utf-8")

    if args.no_eval:
        log("Skipping held-out QA eval (--no-eval).")
        record_status(status_path, "finished", accepted=None)
        return

    log(f"[6/6] Held-out QA eval — RL policy, {len(dev_rows)} questions")
    torch.manual_seed(args.seed)
    rl_eval, rl_records = evaluate_qa_pairs(
        ref, qa, dev_rows, candidates=args.eval_candidates,
        limit=args.eval_questions, max_input=args.max_input,
        max_new=args.max_new, confidence_override=args.conf_override,
        log=log, label="rl")
    record_status(status_path, "eval_done", selected_f1=rl_eval["selected_f1"])

    delta = rl_eval["selected_f1"] - sft_eval["selected_f1"]
    corrective_delta = rl_eval["corrective_f1"] - sft_eval["corrective_f1"]
    consensus_delta = rl_eval["consensus_f1"] - sft_eval["consensus_f1"]
    selected_ci = paired_delta_ci(rl_records, sft_records, "selected_f1")
    corrective_ci = paired_delta_ci(rl_records, sft_records, "corrective_f1")
    consensus_ci = paired_delta_ci(rl_records, sft_records, "consensus_f1")
    stable = (rl_eval["valid_generation_rate"] >= sft_eval["valid_generation_rate"] - 0.05
              and rl_eval["distinct_selected_ratio"] >= 0.50
              and rl_eval["tamil_spill_outputs"] == 0
              and rl_eval["kannada_spill_outputs"] == 0
              and rl_eval["devanagari_spill_outputs"] == 0)
    strict_ok = bool(delta > 0
                     and rl_eval["harmed_rate"] <= sft_eval["harmed_rate"] + 0.05
                     and stable)
    corrective_ok = bool(corrective_delta > 0
                         and rl_eval["corrective_harmed_rate"]
                         <= sft_eval["corrective_harmed_rate"] + 0.05
                         and stable)
    consensus_ok = bool(consensus_delta > 0
                        and rl_eval["consensus_harmed_rate"]
                        <= sft_eval["consensus_harmed_rate"] + 0.05
                        and stable)
    accepted = strict_ok or corrective_ok or consensus_ok
    significant = bool(selected_ci[0] > 0 or corrective_ci[0] > 0
                       or consensus_ci[0] > 0)

    best_policy = "rl" if accepted else start_name
    best_eval = rl_eval if accepted else sft_eval
    selector_scores = {
        "consistent": best_eval["selected_f1"],
        "corrective": best_eval["corrective_f1"],
        "consensus": best_eval["consensus_f1"],
    }
    rule, rule_options = deployable_rule(best_eval)
    best_selector_f1 = (rule_options[rule][0] if rule
                        else max(selector_scores.values()))
    recommended_selector = rule or max(selector_scores,
                                       key=selector_scores.get)
    recommended_action = best_policy if rule else "original"
    if accepted:
        reason = "RL improved the held-out QA objective"
    elif rule:
        reason = ("RL did not beat the starting policy; keep the better "
                  f"{start_name} selector")
    else:
        reason = ("No reformulation policy beat the original question on "
                  "held-out QA with a CI above zero")
    verdict = {
        "accepted": accepted,
        "strict_selection_improved": strict_ok,
        "corrective_selection_improved": corrective_ok,
        "consensus_selection_improved": consensus_ok,
        "significant": significant,
        "recommended_selector": recommended_selector,
        "recommended_action": recommended_action,
        "beats_original": bool(rule),
        "best_policy": best_policy,
        "best_selector_f1": round(best_selector_f1, 4),
        "original_f1": rl_eval["original_f1"],
        "delta": {"selected_f1": round(delta, 4),
                  "corrective_f1": round(corrective_delta, 4),
                  "consensus_f1": round(consensus_delta, 4)},
        "paired_ci95": {"selected_f1": selected_ci,
                        "corrective_f1": corrective_ci,
                        "consensus_f1": consensus_ci},
        "reason": reason,
        "from_base": args.from_base,
        "sft_adapter": str(sft_adapter),
        "rl_adapter": str(adapter_dir),
    }

    print("\n── Held-out QA: original vs selected reformulation ──────────────")
    print(paired_table(sft_eval, rl_eval, start_name.upper()))
    print(f"\n  Consistent selector   RL − SFT selected_f1: {delta:+.4f}   "
          f"paired 95% CI [{selected_ci[0]:+.4f}, {selected_ci[1]:+.4f}]")
    print(f"  Confident correction  RL − SFT corrective_f1: {corrective_delta:+.4f}   "
          f"paired 95% CI [{corrective_ci[0]:+.4f}, {corrective_ci[1]:+.4f}] "
          f"(override ≥ {rl_eval['confidence_override']})")
    print(f"  Answer consensus      RL − SFT consensus_f1: {consensus_delta:+.4f}   "
          f"paired 95% CI [{consensus_ci[0]:+.4f}, {consensus_ci[1]:+.4f}]")
    print(f"  Policy quality (generation oracle): "
          f"{rl_eval['generation_oracle_f1'] - sft_eval['generation_oracle_f1']:+.4f}")
    print("  SFT confidence-margin sweep: " + "  ".join(
        f"{margin}→{row['selected_f1']:.4f}"
        for margin, row in sft_eval["margin_sweep"].items()))
    print(f"  SFT confidence-gate sweep: best q={sft_eval['best_gate']} "
          f"→ {sft_eval['gate_sweep'][sft_eval['best_gate']]['selected_f1']:.4f} "
          f"(original {sft_eval['original_f1']:.4f}); "
          f"ceiling {sft_eval['ceiling_f1']:.4f}")
    print(f"  SFT corrective sweep: best margin "
          f"{sft_eval['best_corrective_margin']} → "
          f"{sft_eval['corrective_sweep'][sft_eval['best_corrective_margin']]['selected_f1']:.4f}  "
          f"| best gate+margin {sft_eval['best_selective']} → "
          f"{sft_eval['selective_sweep'][sft_eval['best_selective']]['selected_f1']:.4f}")
    print(f"  Deployable rule (CI above zero): {rule or 'none'}")
    print(f"  Original question F1: {rl_eval['original_f1']:.4f}   "
          f"best policy+selector: {best_selector_f1:.4f} "
          f"({best_policy}/{recommended_selector})")
    print(f"  Verdict: {verdict['reason']} → use the {best_policy.upper()} "
          f"adapter with the {recommended_selector} selector")
    if recommended_action == "original":
        print("  DEPLOYMENT: do not rewrite. No selector clears the original "
              "question with a CI above zero on this held-out set.")
    if accepted and not significant:
        print("  NOTE: the change is positive but its paired 95% CI includes 0; "
              "rerun with more --questions / --epochs before claiming a gain.")
    if not accepted:
        print("  The RL adapter is kept for inspection; downstream runs should keep "
              "using the starting policy unless you decide otherwise.")

    print("\n── Raw RL examples (no decode-time filtering beyond the policy's "
          "own validity gate) ──")
    for record in rl_records[:3]:
        print(f"\nQ: {record['question']}")
        print(f"   gold: {record['references']}")
        print(f"   original ({record['original_f1']:.2f}): {record['original_answer']}")
        if record["switched"]:
            print(f"   selected ({record['selected_f1']:.2f}): "
                  f"{record['selected_question']}")
        else:
            print("   selected: original question kept")
        for index, (text, f1, valid) in enumerate(zip(
                record["candidate_texts"], record["candidate_f1s"],
                record["candidate_validity"]), 1):
            print(f"     cand {index} ({f1:.2f}, valid={int(valid)}): {text}")

    (out_dir / "qa_eval.json").write_text(json.dumps({
        "sft": sft_eval,
        "rl": rl_eval,
        "delta": {"selected_f1": round(delta, 4),
                  "corrective_f1": round(corrective_delta, 4),
                  "consensus_f1": round(consensus_delta, 4),
                  "generation_oracle_f1": round(
                      rl_eval["generation_oracle_f1"]
                      - sft_eval["generation_oracle_f1"], 4)},
        "paired_ci95": verdict["paired_ci95"],
        "verdict": verdict,
        "training": metrics,
    }, indent=2, ensure_ascii=False), encoding="utf-8")
    (out_dir / "qa_eval_samples.json").write_text(json.dumps({
        "sft": sft_records, "rl": rl_records,
    }, indent=2, ensure_ascii=False), encoding="utf-8")
    record_status(status_path, "finished", accepted=accepted,
                  best_policy=verdict["best_policy"],
                  recommended_action=verdict["recommended_action"],
                  recommended_selector=verdict["recommended_selector"],
                  delta_selected_f1=round(delta, 4))
    log(f"\nDone. Metrics: {out_dir / 'qa_eval.json'}")


if __name__ == "__main__":
    try:
        main()
    except Exception:
        import traceback
        traceback.print_exc()
        log("FAILED with traceback above")
        path = LOG_HANDLE["path"]
        if path:
            try:
                record_status(path.parent / "run_status.json", "failed",
                              error=traceback.format_exc().strip().splitlines()[-1][:300])
            except Exception:
                pass
        raise

In [ ]:
%%writefile /kaggle/working/activeqa/scripts/kaggle_run_all.py
"""One command for the whole Telugu reformulator pipeline on Kaggle/Colab.

    python scripts/kaggle_run_all.py

Stages (each in its own process, so the SFT run's memory is released before RL):

1. scripts/colab_run_sft.py   — install deps, download model, LoRA SFT,
                                base-vs-SFT held-out paraphrase evaluation.
2. scripts/kaggle_run_rl.py   — load the SFT adapter, run on-policy RFT with the
                                frozen Telugu QA model as reward, evaluate the
                                SFT and RL policies on held-out QA, and decide
                                whether the RL adapter is an improvement.

RL without the SFT stage (raw checkpoint + fresh LoRA):
    python scripts/kaggle_run_all.py --from-base

A stronger base checkpoint (any HF seq2seq, prompted — no SFT):
    python scripts/kaggle_run_all.py --model bigscience/mt0-base
    python scripts/kaggle_run_all.py --from-base --model bigscience/mt0-large

Rerun only one stage:
    python scripts/kaggle_run_all.py --from rl
    python scripts/kaggle_run_all.py --from sft
    python scripts/kaggle_run_all.py --reuse-sft          # reuse a saved adapter

Forward stage-specific options with quoted strings:
    python scripts/kaggle_run_all.py --rl-args "--questions 500 --epochs 3"
    python scripts/kaggle_run_all.py --sft-args "--epochs 2 --lr 1.5e-4"

Artifacts:
    models/indicbartss-reformulator-lora/      SFT adapter, sft_metrics.json, run.log
    models/indicbartss-reformulator-lora-rl/   RL adapter, rl_metrics.json,
                                               qa_eval.json, run.log
    models/pipeline_summary.json               both stages in one file
"""
from __future__ import annotations

import argparse
import json
import shlex
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
SFT_DIR = ROOT / "models" / "indicbartss-reformulator-lora"
RL_DIR = ROOT / "models" / "indicbartss-reformulator-lora-rl"


def adapter_ready(directory: Path) -> bool:
    adapter = directory / "adapter"
    return (adapter / "adapter_config.json").is_file() and (
        (adapter / "adapter_model.safetensors").is_file()
        or (adapter / "adapter_model.bin").is_file())


def stage_args(extra: str, add: tuple[str, ...] = ()) -> str:
    """Combine pass-through options without requiring nested shell quotes."""
    parts = shlex.split(extra)
    for option in add:
        if option not in parts:
            parts.append(option)
    return shlex.join(parts)


def run_stage(name: str, script: Path, extra: str) -> int:
    command = [sys.executable, "-u", str(script), *shlex.split(extra)]
    print(f"\n{'=' * 72}\n{name}: {' '.join(command)}\n{'=' * 72}", flush=True)
    started = time.perf_counter()
    code = subprocess.call(command, cwd=str(ROOT))
    print(f"\n{name} finished with exit code {code} in "
          f"{(time.perf_counter() - started) / 60:.1f} min", flush=True)
    return code


def load_json(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None


def main() -> int:
    parser = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--from", dest="stage", default="all",
                        choices=("all", "sft", "rl"),
                        help="start at this stage (default: all)")
    parser.add_argument("--reuse-sft", action="store_true",
                        help="skip SFT when models/indicbartss-reformulator-lora/adapter exists")
    parser.add_argument("--sft-args", default="",
                        help="quoted options for scripts/colab_run_sft.py")
    parser.add_argument("--rl-args", default="",
                        help="options for scripts/kaggle_run_rl.py; use "
                             '--rl-args="--questions 700 --epochs 2" (the equals '
                             "form) so values starting with -- are not parsed "
                             "as wrapper options")
    parser.add_argument("--eval-only", action="store_true",
                        help="run only the SFT-adapter selector diagnostic from "
                             "the RL stage (no training); implies --from rl")
    parser.add_argument("--from-base", action="store_true",
                        help="skip SFT; train RL directly on the raw checkpoint "
                             "with a fresh LoRA (no SFT adapter needed); implies "
                             "--from rl and forwards --from-base to the RL stage")
    parser.add_argument("--model", default=None,
                        help="HF seq2seq checkpoint for from-base RL "
                             "(e.g. bigscience/mt0-base); implies --from-base")
    parser.add_argument("--prompt", default=None,
                        help="override the paraphrase instruction prompt for "
                             "--model checkpoints ({question} placeholder); "
                             "forwarded to the RL stage")
    args = parser.parse_args()
    if args.model:
        if args.stage == "sft":
            parser.error("--model is a from-base RL option; use --from all or --from rl")
        args.stage = "rl"
    if args.from_base:
        if args.stage == "sft":
            parser.error("--from-base skips SFT; use --from all or --from rl")
        args.stage = "rl"
    if args.eval_only:
        if args.stage == "sft":
            parser.error("--eval-only applies to the RL stage; use --from rl")
        args.stage = "rl"

    if args.stage in ("all", "sft"):
        if args.reuse_sft and adapter_ready(SFT_DIR):
            print("Reusing the existing SFT adapter (--reuse-sft).")
        else:
            code = run_stage("STAGE 1 — SFT", ROOT / "scripts" / "colab_run_sft.py",
                             stage_args(args.sft_args))
            if code != 0:
                print("\nSFT failed — stopping before RL. See "
                      f"{SFT_DIR / 'run.log'} for the traceback.")
                return code
            if not adapter_ready(SFT_DIR):
                print("\nSFT exited successfully but produced no adapter. "
                      f"Check {SFT_DIR / 'run_status.json'} and run.log.")
                return 2
            metrics = load_json(SFT_DIR / "sft_metrics.json") or {}
            collapsed = (metrics.get("distinct_best_ratio", 1.0) < 0.5
                         or metrics.get("token_f1", 1.0) == 0.0)
            if collapsed:
                print("\nSFT produced a collapsed adapter (token_f1="
                      f"{metrics.get('token_f1')}, distinct_best_ratio="
                      f"{metrics.get('distinct_best_ratio')}). Stopping before "
                      "RL. Most likely transformers 5.x was active instead of "
                      f"4.57.6; see {SFT_DIR / 'run.log'}.")
                return 3
    if args.stage in ("all", "rl"):
        if not args.from_base and not adapter_ready(SFT_DIR):
            print(f"\nNo SFT adapter at {SFT_DIR / 'adapter'}. "
                  "Run without --from rl (or with --from all) first, or use "
                  "--from-base to train RL straight from the raw checkpoint.")
            return 2
        extra_options: tuple[str, ...] = ()
        if args.eval_only:
            extra_options = (*extra_options, "--eval-only")
        if args.from_base:
            extra_options = (*extra_options, "--from-base")
        if args.model:
            extra_options = (*extra_options, "--model", args.model)
        if args.prompt:
            extra_options = (*extra_options, "--rl-prompt", args.prompt)
        code = run_stage("STAGE 2 — RL", ROOT / "scripts" / "kaggle_run_rl.py",
                         stage_args(args.rl_args, extra_options))
        if code != 0:
            print("\nRL failed. The SFT adapter is still valid; see "
                  f"{RL_DIR / 'run.log'} for the traceback.")
            return code

    sft_metrics = load_json(SFT_DIR / "sft_metrics.json")
    qa_path = RL_DIR / "qa_eval.json"
    sft_only_path = RL_DIR / "qa_eval_sft_only.json"
    eval_only = (sft_only_path.exists()
                 and (not qa_path.exists()
                      or sft_only_path.stat().st_mtime
                      > qa_path.stat().st_mtime))
    rl_eval = None if eval_only else load_json(qa_path)
    sft_only = load_json(sft_only_path) if eval_only else None
    summary = {
        "sft_metrics": sft_metrics,
        "qa_eval": rl_eval,
        "sft_selector_diagnostics": sft_only,
        "artifacts": {
            "sft_adapter": str(SFT_DIR / "adapter"),
            "rl_adapter": str(RL_DIR / "adapter"),
        },
    }
    (ROOT / "models").mkdir(parents=True, exist_ok=True)
    (ROOT / "models" / "pipeline_summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

    print("\n" + "=" * 72)
    if sft_metrics:
        print(f"SFT paraphrase token-F1: "
              f"{sft_metrics.get('baseline', {}).get('token_f1', float('nan')):.4f} "
              f"→ {sft_metrics.get('token_f1', float('nan')):.4f}   "
              f"(optimizer steps {sft_metrics.get('optimizer_steps', '?')})")
    if sft_only:
        ev = sft_only["sft"]
        print("Selector-only diagnostic (no RL training this run):")
        print(f"  original {ev['original_f1']:.4f} | "
              f"consistent {ev['selected_f1']:.4f} | "
              f"corrective {ev['corrective_f1']:.4f} | "
              f"consensus {ev['consensus_f1']:.4f}")
        print(f"  ceiling (original + rewrites) {ev['ceiling_f1']:.4f} "
              f"({ev['ceiling_delta_f1']:+.4f}); "
              f"recoverable {ev['recoverable_rate']:.3f}")
        best_margin = ev["margin_sweep"][ev["best_margin"]]["selected_f1"]
        print(f"  best confidence margin on dev: {ev['best_margin']} "
              f"({best_margin:.4f}) — diagnostic, not a deployment claim")
        rules = {
            "consistent": (ev["selected_f1"], ev.get("delta_ci95")),
            "corrective": (ev["corrective_f1"],
                           ev.get("corrective_delta_ci95")),
            "consensus": (ev["consensus_f1"],
                          ev.get("consensus_delta_ci95")),
        }
        for name, row in ev.get("corrective_sweep", {}).items():
            rules[f"corrective@{name}"] = (row["selected_f1"],
                                           row.get("delta_ci95"))
        for name, row in ev.get("selective_sweep", {}).items():
            rules[f"selective:{name}"] = (row["selected_f1"],
                                          row.get("delta_ci95"))
        deployable = {name: value for name, value in rules.items()
                      if value[0] > ev["original_f1"] + 1e-9
                      and value[1] and value[1][0] > 0}
        if deployable:
            best_rule = max(deployable,
                            key=lambda name: deployable[name][0])
            print(f"  deployable rule: {best_rule} "
                  f"({deployable[best_rule][0]:.4f}, "
                  f"CI[{deployable[best_rule][1][0]:+.4f},"
                  f"{deployable[best_rule][1][1]:+.4f}])")
        else:
            print("  deployable rule: none — no rule clears the original with a "
                  "CI above zero")
            print("  DEPLOYMENT: answer the original question; do not rewrite.")
    if rl_eval:
        verdict = rl_eval["verdict"]
        print(f"RL held-out QA selected-F1: "
              f"{rl_eval['sft']['selected_f1']:.4f} → "
              f"{rl_eval['rl']['selected_f1']:.4f} "
              f"({rl_eval['delta']['selected_f1']:+.4f})")
        if verdict.get("recommended_action") == "original":
            print(f"Verdict: {verdict['reason']} → answer the original "
                  f"question; no {verdict['best_policy'].upper()} selector is "
                  f"deployable")
        else:
            print(f"Verdict: {verdict['reason']} → "
                  f"{verdict['recommended_action'].upper()} "
                  f"({verdict['best_policy']} adapter, "
                  f"{verdict['recommended_selector']} selector)")
    print(f"Summary: {ROOT / 'models' / 'pipeline_summary.json'}")
    print("Zip models/indicbartss-reformulator-lora and "
          "models/indicbartss-reformulator-lora-rl to download every artifact.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
import os
import shutil
from huggingface_hub import hf_hub_download
import pathlib
DATA_DIR = pathlib.Path("/kaggle/working/activeqa/data/indic_rag_te")
DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in ("train.jsonl", "dev.jsonl"):
    p = hf_hub_download(repo_id="nlpctx/activeqa-telugu-qa", filename=name,
                        repo_type="dataset", token=os.environ.get("HF_TOKEN"))
    shutil.copy(p, DATA_DIR / name)
print("dataset ready:", DATA_DIR)


In [ ]:
!cd /kaggle/working/activeqa && python scripts/kaggle_run_all.py --from-base


In [ ]:
import os
token = os.environ.get("HF_TOKEN")
if not token:
    raise RuntimeError("set os.environ['HF_TOKEN'] in the first cell, e.g. os.environ['HF_TOKEN'] = 'hf_...'")
import shutil
import sys
from pathlib import Path
ROOT = Path("/kaggle/working/activeqa")
sys.path.insert(0, str(ROOT / "src"))
BASE = ROOT / "models" / "MultiIndicParaphraseGenerationSS"
ADAPTER = ROOT / "models" / "indicbartss-reformulator-lora-rl" / "adapter"
OUT = ROOT / "models" / "hf-push"
if not ADAPTER.is_dir():
    raise RuntimeError(f"RL adapter missing: {ADAPTER} - run the training cell first")
from huggingface_hub import HfApi, login
login(token=token)
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
model = AutoModelForSeq2SeqLM.from_pretrained(str(BASE))
model = PeftModel.from_pretrained(model, str(ADAPTER))
merged = model.merge_and_unload()
tok = AutoTokenizer.from_pretrained(str(BASE), use_fast=False, do_lower_case=False, keep_accents=True)
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)
merged.save_pretrained(str(OUT))
tok.save_pretrained(str(OUT))
card_lines = [
    "---",
    "language: te",
    "license: mit",
    "tags:",
    "- question-reformulation",
    "- telugu",
    "- reinforcement-learning",
    "- lora",
    "- activeqa",
    "---",
    "",
    "# ActiveQATeluguReformulatorRL (merged LoRA)",
    "",
    "MultiIndicParaphraseGenerationSS post-trained with a validation-gated RL loop",
    "(entity-preservation reward, QA reward l3cube-pune/telugu-question-answering-squad-bert).",
    "No SFT: the LoRA trained directly from the raw checkpoint on 804 Telugu questions,",
    "best checkpoint selected on 200 held-out dev questions.",
    "",
    "Dataset: ai4bharat/Indic-Rag-Suite te sample (LLM-generated QA from Telugu",
    "Wikipedia). Answers are silver and not human-verified. Research artifact;",
    "not a production QA system.",
]
(OUT / "README.md").write_text("\n".join(card_lines) + "\n")
api = HfApi(token=token)
api.create_repo(repo_id="nlpctx/activeqa-telugu-reformulator-rl", exist_ok=True)
api.upload_folder(repo_id="nlpctx/activeqa-telugu-reformulator-rl", folder_path=str(OUT), repo_type="model")
print("pushed https://huggingface.co/nlpctx/activeqa-telugu-reformulator-rl")


In [ ]:
import json
import random
import sys
from pathlib import Path
ROOT = Path("/kaggle/working/activeqa")
sys.path.insert(0, str(ROOT / "src"))
import torch
from activeqa.reformulator import Reformulator
from activeqa.qa_model import TeluguQA
from activeqa.preprocessing import normalize
from activeqa.rewards import token_f1
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); random.seed(42)
ref = Reformulator("nlpctx/activeqa-telugu-reformulator-rl",
                   cache_dir="/kaggle/working/hf_demo_cache", device=device)
qa = TeluguQA("l3cube-pune/telugu-question-answering-squad-bert",
              cache_dir=str(ROOT / "models"), device=device)
rows = [json.loads(l) for l in open(ROOT / "data/indic_rag_te/dev.jsonl", encoding="utf-8")][:3]
for row in rows:
    q = normalize(row["question"])
    ctx = row["context"]
    gold = row["answers"]
    ans_q, _, score_q = qa.answer_with_score(q, ctx)
    acts = ref.sample_actions(q, n=6, max_input=96, max_new=64, do_sample=True)
    valid = [a for a in acts if a["valid"]]
    print("=" * 78)
    print("Q :", q)
    print("   gold:", gold)
    print(f"   ANSWER(original) [{score_q:.2f}]: {ans_q}   f1={token_f1(ans_q, gold):.2f}")
    if not valid:
        print("   (all candidates invalid)")
    for i, a in enumerate(valid, 1):
        ans, _, s = qa.answer_with_score(a["candidate"], ctx)
        print(f"   cand {i} (valid): {a['candidate']}")
        print(f"      ANSWER [{s:.2f}]: {ans}   f1={token_f1(ans, gold):.2f}")
print("=" * 78)
print("valid-rate on last question:", sum(a["valid"] for a in acts), "/", len(acts))
